# Consolidação e Calibração de Base de Matrículas

> **Portfolio pessoal.** Este notebook foi extraído de um projeto real de automação de processos acadêmicos e passou por um processo de anonimização: nomes de instituições, pessoas e caminhos locais foram substituídos por termos genéricos, e todos os outputs de execução (que continham dados reais de alunos/professores) foram removidos. O código e a lógica de negócio foram mantidos como estavam em produção.

Consolida planilhas de fluxo, mapa pedagógico e registro de professores em uma base única calibrada, com auditoria de duplicatas e diagnóstico de qualidade.

---


# Automação de Base Acadêmica — versão final formatada 51/52/53 e preparação para 54

Notebook operacional para gerar a base do dashboard a partir de **FLUXO**, **MAPA PEDAGÓGICO** e **REGISTRO DE PROFESSORES**.

Esta versão mantém a calibração de alunos validada, verifica duplicatas integrais sem remover, preserva os valores de mediadores da base manual/revisada e exporta as abas de dados com a mesma formatação visual da planilha preenchida manualmente.


## Revisão de calibração de alunos

Esta versão trava os módulos históricos pela **base conferida embutida** quando não houver arquivo de base manual na pasta. Isso evita a regressão em que o módulo 51 era recalculado pela automação bruta e perdia alunos em relação à base validada manualmente.

Para o módulo 54, você pode continuar usando apenas os três arquivos novos (`FLUXO`, `MAPA`, `REGISTRO`): mantenha `USAR_BASE_CONFERIDA=True` e `USAR_BASE_CONFERIDA_EMBUTIDA=True`, defina `MODULOS_SAIDA = [51, 52, 53, 54]` e `MODULOS_PROCESSAR_AUTO = [54]`.

## Guia rápido de uso

1. Coloque os arquivos na mesma pasta do notebook:
   - `FLUXO*.xlsx`
   - `MAPA PEDAGÓGICO <módulo>-2026*.xlsx`
   - `REGISTRO_PROFESSORES*.xlsx`
   - opcionalmente, a base conferida mais recente, por exemplo `Base_Instituicao_515253_revisada_alunos.xlsx`.
2. Ajuste os parâmetros na seção **2. Parâmetros**.
3. Rode todas as células.
4. Use a aba `GERAL` como fonte principal do dashboard.
5. Consulte `DIAGNOSTICOS` para entender se a busca no mapa/fluxo/monitoramento foi exata ou relaxada.

> Atenção: a partir desta versão existem dois parâmetros diferentes. `MODULOS_SAIDA` define o que deve aparecer no Excel final. `MODULOS_PROCESSAR_AUTO` define quais mapas serão processados automaticamente nesta execução. Para manter o histórico 51/52/53 e acrescentar o 54, use `MODULOS_SAIDA = [51, 52, 53, 54]` e `MODULOS_PROCESSAR_AUTO = [54]`.


## Como a automação funciona

A automação replica o processo manual com uma camada de auditoria:

1. **Registro**: a aba `PREENCHIMENTO|LETIVAS` define quais disciplinas/aulas devem existir na base, com professor, chapa, carga horária, módulo, ordem, dia e horário.
2. **Mapa pedagógico**: para cada disciplina do registro, o notebook procura as turmas/curso que participam da mesma aula usando a chave **disciplina + ordem + dia + horário**. Quando esses quatro campos batem, os cursos são considerados sinérgicos.
3. **Fluxo**: calcula `QTD_ALUNOS` somando o número de alunos do fluxo por **ano + módulo/semestre + curso + disciplina**.
4. **Monitoramento**: busca mediador e audiência média nas abas de monitoramento do mapa.
5. **Calibração com base manual**: módulos já conferidos manualmente não são substituídos pela automação. A base manual prevalece e a automação serve para diagnóstico/incremento.
6. **Sinergia**: preenche `CURSOS_SINERGIA` e `CLUSTERS_SINERGIA` com base nos cursos que compartilham a mesma chave de aula.
7. **Diagnósticos**: registra o método de busca usado e os pontos que precisam de revisão.


## Regras de negócio já incorporadas

- Os módulos **51**, **52** e **53** podem ser tratados como módulos conferidos quando a base revisada estiver disponível.
- A base conferida tem prioridade sobre a automação nos módulos fechados.
- Para módulos novos, como o **54**, a automação gera as linhas a partir de registro + mapa + fluxo + monitoramento.
- O cruzamento principal do mapa usa: **disciplina + ordem + dia + horário**.
- Quando a base conferida tiver a aba `GERAL`, ela é lida como fonte única para evitar duplicidade com as abas `BASE5126`, `BASE5226`, etc.
- A exceção `PROJETO EXPERIMENTAL` permanece auditável, com 80 alunos conforme conferência manual no fluxo.

- Antes da exportação, o notebook verifica **duplicatas integrais** na base final. Quando encontra linhas 100% iguais, **não remove nada**: mantém a base conforme o gabarito/manual e registra o detalhe na aba `DIAGNOSTICOS`.


## Correção técnica — comparação de chaves

Esta versão corrige o erro `ValueError: Lengths must be equal` que podia ocorrer na aplicação dos ajustes manuais de alunos.

A causa era uma comparação direta entre uma `Series` de tuplas e uma tupla de chave. Agora a comparação é feita linha a linha, mantendo a lógica validada e evitando interpretação indevida pelo pandas.


## Revisão de formatação e mediadores

Antes da exportação, o notebook agora valida as colunas `MEDIADOR`, `CHAPA2`, `CH_SEMANAL` e `MEDIA_AUDIENCIA`.

A prioridade é: primeiro a planilha manual `Base_Instituicao_515253.xlsx`, depois a versão revisada anterior, e por último uma normalização de segurança para evitar a regressão em que `CH_SEMANAL` e `MEDIA_AUDIENCIA` eram gravados multiplicados por 10.

Na saída, `CHAPA` e `CHAPA2` são gravadas como texto para preservar zeros à esquerda, e as abas `GERAL`/`BASEXXXX` recebem o padrão visual verde da planilha manual.


## Interpretação das principais abas geradas

| Aba | Finalidade |
|---|---|
| `GERAL` | Base consolidada com todos os módulos processados. É a principal fonte para o dashboard. |
| `BASE5126`, `BASE5226`, etc. | Recortes por módulo, úteis para conferência isolada. |
| `DIAGNOSTICOS` | Mostra como cada linha foi encontrada e se houve busca exata, relaxada, sem correspondência, ajuste manual ou remoção de duplicata integral. |
| `INCREMENTOS_AUTOMATICOS` | Linhas que a automação adicionou a uma base manual parcialmente conferida. |
| `COMPARATIVO_CALIBRACAO` | Comparação entre base manual, automação bruta e base final. |
| `DIVERGENCIAS_AUTO_MANUAL` | Linhas em que a automação divergiria da base manual. Serve para auditoria, não para substituir a base conferida. |
| `DIAG_SINERGIA` | Linhas que ainda teriam sinergia incompleta antes dos ajustes. |
| `RESUMO_REVISAO` | Resumo final de exportação, módulos e ajustes aplicados. |
| `DUPLICATAS_IDENTIFICADAS` | Relação das linhas inteiras duplicadas encontradas. Elas são mantidas na base final; a aba serve apenas para auditoria. |
| `CONTROLE_MEDIADORES` | Mostra as linhas em que valores de mediadores foram preservados ou normalizados antes da exportação. |


## Validação adicional: R$ (RPA) e formatação final

Esta versão também preserva a coluna `R$ (RPA)` da base manual/gabarito. A validação impede a regressão em que os valores do módulo 51 eram gravados multiplicados por 10.

Antes da exportação, o notebook cruza as linhas pela mesma chave de negócio usada na base e substitui o valor de `R$ (RPA)` pelo valor conferido quando a linha existe na base manual. Linhas novas, como complementos do módulo 53 ou futuro 54, têm o valor convertido de forma segura, sem alterar casas decimais.


## Correção específica Power BI: RPA do módulo 53

A aba `BASE5326` mantém a mesma estrutura das abas `BASE5126` e `BASE5226`. A inconsistência identificada estava no preenchimento de `R$ (RPA)`: quatro linhas do 53 já existiam na base manual parcial e, por isso, tiveram A:S preservadas antes da atualização do registro. Nesta versão, o notebook preserva o gabarito manual de 51/52 e sincroniza o RPA do 53 diretamente pelo `REGISTRO`, usando a chave `MÓDULO + DISCIPLINA + ORDEM + DIA + HORÁRIO`.


## Preservação da base manual

Para os módulos já conferidos, as colunas A:S da base manual são tratadas como gabarito. A automação só complementa/enriquece campos derivados, como sinergia, diagnósticos e módulos novos.


## 1. Funções da automação

As funções abaixo padronizam textos, detectam abas, fazem os cruzamentos e exportam o arquivo final. Normalmente você não precisa alterar esta célula.

In [ ]:
from pathlib import Path
import re
import unicodedata
from collections import Counter
import warnings

import numpy as np
import pandas as pd

COLUNAS_BASE = [
    'CLUSTER','CURSO','DISCIPLINA','MÓDULO','ORDEM','DIA','HORÁRIO','QTD_ALUNOS','MODULAÇÃO',
    'PROFESSOR','CHAPA','CARGO','CH SEMANAL','CH LETIVA','R$ (RPA)',
    'MEDIADOR','CHAPA2','CH_SEMANAL','MEDIA_AUDIENCIA','CURSOS_SINERGIA','CLUSTERS_SINERGIA'
]

DIAS_PREFIXO = {
    'SEGUNDA': '2 - Segunda',
    'TERCA': '3 - Terça',
    'TERÇA': '3 - Terça',
    'QUARTA': '4 - Quarta',
    'QUINTA': '5 - Quinta',
    'SEXTA': '6 - Sexta',
    'SABADO': '7 - Sábado',
    'SÁBADO': '7 - Sábado',
    'DOMINGO': '1 - Domingo',
}

HORARIOS_PADRAO = {
    '19:00': '19:00 às 20:30',
    '20:40': '20:40 às 22:10',
}


def normalizar_texto(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r'\s+', ' ', texto)
    return texto


def texto_limpo(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip()
    texto = re.sub(r'\s+', ' ', texto)
    return texto


def normalizar_ordem(valor):
    s = normalizar_texto(valor)
    if not s or s in {'NAN','NONE','NSA'}:
        return ''
    if 'ESTEND' in s:
        return 'ESTENDIDA'
    if re.search(r'(^|\D)1(\D|$)', s) or 'PRIMEIR' in s:
        return '1'
    if re.search(r'(^|\D)2(\D|$)', s) or 'SEGUND' in s:
        return '2'
    return s


def ordem_saida(valor):
    k = normalizar_ordem(valor)
    if k in {'1','2','ESTENDIDA'}:
        return k
    return texto_limpo(valor)


def normalizar_dia(valor):
    s = normalizar_texto(valor)
    if not s or s in {'NAN','NONE','NSA'}:
        return ''
    for chave, saida in DIAS_PREFIXO.items():
        if normalizar_texto(chave) in s:
            return normalizar_texto(saida)
    return s


def dia_saida(valor):
    k = normalizar_dia(valor)
    mapa = {normalizar_texto(v): v for v in DIAS_PREFIXO.values()}
    return mapa.get(k, texto_limpo(valor))


def normalizar_horario(valor):
    if pd.isna(valor):
        return ''
    if hasattr(valor, 'hour') and hasattr(valor, 'minute'):
        return f'{int(valor.hour):02d}:{int(valor.minute):02d}'
    texto = str(valor).strip().lower()
    if not texto or texto in {'nan','none','nat','nsa'}:
        return ''
    m = re.search(r'(\d{1,2})\s*[:h]\s*(\d{2})?', texto)
    if m:
        return f'{int(m.group(1)):02d}:{int(m.group(2) or 0):02d}'
    return normalizar_texto(texto)


def horario_saida(valor):
    h = normalizar_horario(valor)
    if h in HORARIOS_PADRAO:
        return HORARIOS_PADRAO[h]
    return texto_limpo(valor)


def extrair_modulo_num(valor):
    s = str(valor)
    # Captura 51, 52, 53, 54 etc. evitando ano 2026.
    m = re.search(r'(?<!\d)(5\d)(?!\d)', s)
    return int(m.group(1)) if m else None


def encontrar_linha_cabecalho(caminho, aba, marcadores):
    bruto = pd.read_excel(caminho, sheet_name=aba, header=None, nrows=40, dtype=str)
    marcadores_norm = [normalizar_texto(x) for x in marcadores]
    for idx, row in bruto.iterrows():
        valores = {normalizar_texto(v) for v in row.dropna().tolist()}
        if all(m in valores for m in marcadores_norm):
            return idx
    return 0


def encontrar_arquivo(pasta, padroes, obrigatorio=True):
    pasta = Path(pasta)
    candidatos = []
    for padrao in padroes:
        candidatos.extend(pasta.glob(padrao))
    candidatos = [p for p in candidatos if p.is_file() and not p.name.startswith('~$')]
    candidatos = sorted(set(candidatos), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidatos:
        return candidatos[0]
    if obrigatorio:
        raise FileNotFoundError(f'Não encontrei arquivo com padrões: {padroes} em {pasta}')
    return None


def localizar_mapas(pasta, modulos_processar=None):
    pasta = Path(pasta)
    if modulos_processar is not None and len(modulos_processar) == 0:
        return []
    mapas = [p for p in pasta.glob('MAPA PEDAGÓGICO*.xlsx') if p.is_file() and not p.name.startswith('~$')]
    if not mapas:
        mapas = [p for p in pasta.glob('MAPA PEDAGOGICO*.xlsx') if p.is_file() and not p.name.startswith('~$')]
    if modulos_processar:
        mods = {int(m) for m in modulos_processar}
        mapas = [p for p in mapas if extrair_modulo_num(p.name) in mods]

    # Quando há versões antigas e novas na mesma pasta, seleciona o arquivo mais
    # recente de cada módulo. Isso evita reprocessar duas cópias do mesmo mapa.
    por_modulo = {}
    sem_modulo = []
    for p in sorted(mapas, key=lambda p: p.stat().st_mtime, reverse=True):
        modulo = extrair_modulo_num(p.name)
        if modulo is None:
            sem_modulo.append(p)
        elif modulo not in por_modulo:
            por_modulo[modulo] = p
    mapas = [por_modulo[m] for m in sorted(por_modulo)] + sem_modulo

    if not mapas:
        raise FileNotFoundError('Não encontrei mapas pedagógicos para os módulos selecionados.')
    return mapas


def carregar_registro(caminho_registro):
    header = encontrar_linha_cabecalho(caminho_registro, 'PREENCHIMENTO|LETIVAS', ['PROFESSOR(A)', 'DISCIPLINA (NOME)'])
    df = pd.read_excel(caminho_registro, sheet_name='PREENCHIMENTO|LETIVAS', header=header, dtype=str)
    df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]
    obrig = ['PROFESSOR(A)', 'CHAPA', 'CH SEMANAL', 'CH LETIVA', 'CARGO', 'MÓDULO', 'ORDEM', 'DIA DA TRANSMISSÃO', 'HORÁRIO', 'DISCIPLINA (NOME)', 'OBSERVAÇÃO', 'R$ (RPA)']
    for c in obrig:
        if c not in df.columns:
            df[c] = np.nan
    df = df.dropna(subset=['MÓDULO', 'DISCIPLINA (NOME)'], how='any').copy()
    df['modulo_num'] = df['MÓDULO'].map(extrair_modulo_num)
    df['disc_key'] = df['DISCIPLINA (NOME)'].map(normalizar_texto)
    df['ord_key'] = df['ORDEM'].map(normalizar_ordem)
    df['day_key'] = df['DIA DA TRANSMISSÃO'].map(normalizar_dia)
    df['hour_key'] = df['HORÁRIO'].map(normalizar_horario)
    chave = ['PROFESSOR(A)','CHAPA','CARGO','MÓDULO','ORDEM','DIA DA TRANSMISSÃO','HORÁRIO','DISCIPLINA (NOME)','OBSERVAÇÃO']
    df = df.drop_duplicates(subset=chave, keep='first').reset_index(drop=True)
    return df


def carregar_fluxo(caminho_fluxo, ano=2026):
    df = pd.read_excel(caminho_fluxo, dtype={'CURSO':str,'NOME_COMPL':str})
    for c in ['ANO','SEMESTRE','QTDE_ALUNOS']:
        if c not in df.columns:
            raise ValueError(f'Coluna obrigatória ausente no fluxo: {c}')
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df[df['ANO'].eq(ano)].copy()
    df['curso_key'] = df['CURSO'].map(normalizar_texto)
    df['disc_key'] = df['NOME_COMPL'].map(normalizar_texto)
    df = df.dropna(subset=['SEMESTRE'])
    agg = (df.groupby(['SEMESTRE','curso_key','disc_key'], as_index=False)['QTDE_ALUNOS']
             .sum())
    agg['SEMESTRE'] = agg['SEMESTRE'].astype(int)
    return agg


def detectar_aba_mapa(caminho_mapa):
    xls = pd.ExcelFile(caminho_mapa)
    candidatas = []
    ignorar = ['MONITORAMENTO','ESTAGIO','ESTÁGIO','TCC','CALENDARIO','CALENDÁRIO','DIAS','MODELOS','AGENDA','PLANEJ','DOCENTES','LOG','RECURSOS','TAB_DIN','INSTITUCIONAIS']
    for aba in xls.sheet_names:
        nome_norm = normalizar_texto(aba)
        if any(t in nome_norm for t in ignorar):
            continue
        try:
            amostra = pd.read_excel(caminho_mapa, sheet_name=aba, nrows=5, dtype=str)
        except Exception:
            continue
        cols = {normalizar_texto(c) for c in amostra.columns}
        tem_disciplina = bool({'NOME DISCIPLINA','DISCIPLINA'} & cols)
        if {'CURSO','ORDEM','DIA AULA','HORARIO'}.issubset(cols) and tem_disciplina:
            candidatas.append(aba)
    if not candidatas:
        raise ValueError(f'Não encontrei uma aba principal de mapa em {caminho_mapa}.')
    candidatas = sorted(candidatas, key=lambda x: ('COPIA' in normalizar_texto(x), len(x)))
    return candidatas[0]


def detectar_aba_monitoramento(caminho_mapa):
    xls = pd.ExcelFile(caminho_mapa)
    candidatas = [s for s in xls.sheet_names if 'MONITORAMENTO' in normalizar_texto(s)]
    return candidatas[0] if candidatas else None


def carregar_mapa(caminho_mapa):
    aba = detectar_aba_mapa(caminho_mapa)
    df = pd.read_excel(caminho_mapa, sheet_name=aba, dtype=str)
    df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]
    col_disc = 'NOME DISCIPLINA' if 'NOME DISCIPLINA' in df.columns else 'DISCIPLINA'
    for c in ['CLUSTER', 'SINERGIA', 'CURSO', 'ORDEM', 'DIA AULA', 'HORÁRIO']:
        if c not in df.columns:
            df[c] = np.nan
    # Alguns mapas vêm com CLUSTER mesclado por bloco; o pandas enxerga só a primeira linha.
    # O ffill preserva o agrupamento sem alterar mapas que já vêm linha a linha.
    if 'CLUSTER' in df.columns:
        df['CLUSTER'] = df['CLUSTER'].ffill()
    df['disciplina_nome'] = df[col_disc]
    if 'MÓD' in df.columns:
        df['modulo_num'] = df['MÓD'].map(extrair_modulo_num)
    else:
        df['modulo_num'] = extrair_modulo_num(aba)
    if not isinstance(df['modulo_num'], pd.Series):
        df['modulo_num'] = df['modulo_num']
    if pd.Series(df['modulo_num']).isna().all():
        df['modulo_num'] = extrair_modulo_num(aba)
    df['curso_key'] = df['CURSO'].map(normalizar_texto)
    df['disc_key'] = df['disciplina_nome'].map(normalizar_texto)
    df['ord_key'] = df['ORDEM'].map(normalizar_ordem)
    df['day_key'] = df['DIA AULA'].map(normalizar_dia)
    df['hour_key'] = df['HORÁRIO'].map(normalizar_horario)
    df = df[df['disc_key'].ne('') & df['curso_key'].ne('')].copy()
    return aba, df


def carregar_monitoramento(caminho_mapa):
    aba = detectar_aba_monitoramento(caminho_mapa)
    if not aba:
        return None, pd.DataFrame()
    df = pd.read_excel(caminho_mapa, sheet_name=aba, dtype=str)
    df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]
    required = ['CURSO','DISCIPLINA','ORDEM','DIA DA SEMAMA','HORÁRIO','AUDIÊNCIA']
    for c in required:
        if c not in df.columns:
            df[c] = np.nan
    nome_cols = [c for c in df.columns if str(c).startswith('NOME')]
    df['MEDIADOR_RAW'] = df[nome_cols[1]] if len(nome_cols) >= 2 else np.nan
    df['curso_key'] = df['CURSO'].map(normalizar_texto)
    df['disc_key'] = df['DISCIPLINA'].map(normalizar_texto)
    df['ord_key'] = df['ORDEM'].map(normalizar_ordem)
    df['day_key'] = df['DIA DA SEMAMA'].map(normalizar_dia)
    df['hour_key'] = df['HORÁRIO'].map(normalizar_horario)
    df['AUDIÊNCIA_NUM'] = pd.to_numeric(df['AUDIÊNCIA'], errors='coerce')
    df['MEDIADOR_RAW_LIMPO'] = df['MEDIADOR_RAW'].map(texto_limpo)
    return aba, df


def escolher_representante(candidatos):
    if candidatos is None or candidatos.empty:
        return None
    sin = candidatos['SINERGIA'].map(normalizar_texto) if 'SINERGIA' in candidatos.columns else pd.Series('', index=candidatos.index)
    resp = candidatos[sin.str.contains('RESPONS', na=False)]
    if not resp.empty:
        return resp.iloc[0]
    unico = candidatos[sin.str.contains('CURSO UNICO|CURSO ÚNICO', na=False)]
    if not unico.empty:
        return unico.iloc[0]
    return candidatos.iloc[0]


def unir_unicos(series):
    vals = [texto_limpo(v) for v in series.dropna().tolist()]
    vals = [v for v in vals if v and normalizar_texto(v) not in {'NAN','NONE','NSA'}]
    vistos, out = set(), []
    for v in vals:
        k = normalizar_texto(v)
        if k not in vistos:
            out.append(v)
            vistos.add(k)
    return ' | '.join(out)


def selecionar_candidatos_mapa(mapa, linha_registro):
    """Retorna candidatos do mapa para uma aula.

    A chave principal é disciplina + ordem + dia + horário. Quando dia ou horário vêm vazios no registro,
    a rotina relaxa a chave e usa o mapa para preencher a aula mais provável. Se o relaxamento trouxer mais
    de um horário/dia, ela restringe os cursos à aula do representante escolhido para evitar somar aulas distintas.
    """
    disc = linha_registro['disc_key']
    ordem = linha_registro['ord_key']
    dia = linha_registro['day_key']
    hora = linha_registro['hour_key']

    base = mapa['disc_key'].eq(disc)
    if ordem:
        base = base & mapa['ord_key'].eq(ordem)

    tentativas = []
    if dia and hora:
        tentativas.append(('exata: disciplina+ordem+dia+horário', base & mapa['day_key'].eq(dia) & mapa['hour_key'].eq(hora)))
    if dia:
        tentativas.append(('relaxada: disciplina+ordem+dia', base & mapa['day_key'].eq(dia)))
    if hora:
        tentativas.append(('relaxada: disciplina+ordem+horário', base & mapa['hour_key'].eq(hora)))
    tentativas.append(('relaxada: disciplina+ordem', base))
    tentativas.append(('relaxada: somente disciplina', mapa['disc_key'].eq(disc)))

    for tipo, mask in tentativas:
        cand = mapa[mask].copy()
        if not cand.empty:
            rep = escolher_representante(cand)
            cand_aula = cand.copy()
            if rep is not None:
                rep_day = rep.get('day_key', '')
                rep_hour = rep.get('hour_key', '')
                # Sempre que a busca foi relaxada, mantém somente a aula do representante para não misturar horários/dias.
                if not dia and rep_day:
                    cand_aula = cand_aula[cand_aula['day_key'].eq(rep_day)]
                if not hora and rep_hour:
                    cand_aula = cand_aula[cand_aula['hour_key'].eq(rep_hour)]
                if tipo != 'exata: disciplina+ordem+dia+horário':
                    if rep_day:
                        cand_aula = cand_aula[cand_aula['day_key'].eq(rep_day)]
                    if rep_hour:
                        cand_aula = cand_aula[cand_aula['hour_key'].eq(rep_hour)]
                if cand_aula.empty:
                    cand_aula = cand
                    rep = escolher_representante(cand_aula)
                else:
                    rep = escolher_representante(cand_aula)
            return cand_aula, rep, tipo
    return pd.DataFrame(columns=mapa.columns), None, 'sem correspondência no mapa'



def construir_lookup_mediadores(base_historica, monitoramentos_por_modulo):
    """Cria dicionário raw->dados do mediador usando base conferida.

    A função usa o mesmo carregador de gabarito da calibração. Assim, quando não houver
    arquivo externo de base conferida na pasta, a base embutida ainda pode enriquecer
    mediadores dos módulos históricos.
    """
    lookup = {}
    try:
        base = carregar_base_conferida(base_historica)
    except Exception:
        base = pd.DataFrame()
    if base.empty:
        return lookup
    base = base.dropna(subset=['MEDIADOR']) if 'MEDIADOR' in base.columns else pd.DataFrame()
    if base.empty:
        return lookup
    base['modulo_num'] = base['MÓDULO'].map(extrair_modulo_num)
    base['disc_key'] = base['DISCIPLINA'].map(normalizar_texto)
    base['ord_key'] = base['ORDEM'].map(normalizar_ordem)
    base['day_key'] = base['DIA'].map(normalizar_dia)
    base['hour_key'] = base['HORÁRIO'].map(normalizar_horario)
    for _, row in base.iterrows():
        mod = row['modulo_num']
        mon = monitoramentos_por_modulo.get(mod)
        if mon is None or mon.empty:
            continue
        m = mon[(mon['disc_key'].eq(row['disc_key'])) & (mon['ord_key'].eq(row['ord_key'])) & (mon['day_key'].eq(row['day_key'])) & (mon['hour_key'].eq(row['hour_key']))]
        for raw in m['MEDIADOR_RAW_LIMPO'].dropna().unique():
            raw_key = normalizar_texto(raw)
            if raw_key:
                lookup[raw_key] = {
                    'MEDIADOR': row.get('MEDIADOR'),
                    'CHAPA2': row.get('CHAPA2'),
                    'CH_SEMANAL': row.get('CH_SEMANAL'),
                }
    return lookup

def enriquecer_mediador(raw, lookup):
    raw = texto_limpo(raw)
    if not raw:
        return '', '', ''
    partes = re.split(r'\s*/\s*|\s+E\s+|\s*\|\s*', raw)
    nomes, chapas, chs = [], [], []
    for p in partes:
        key = normalizar_texto(p)
        if not key:
            continue
        dado = lookup.get(key)
        if dado:
            nomes.append(texto_limpo(dado.get('MEDIADOR')))
            chapas.append(texto_limpo(dado.get('CHAPA2')))
            chs.append(texto_limpo(dado.get('CH_SEMANAL')))
        else:
            nomes.append(texto_limpo(p).upper())
    return unir_unicos(pd.Series(nomes)), unir_unicos(pd.Series(chapas)), unir_unicos(pd.Series(chs))


def montar_linha_monitoramento(mon, r, cursos, lookup_mediadores):
    mon_match = pd.DataFrame()
    if mon is not None and not mon.empty:
        mon_match = mon[(mon['disc_key'].eq(r['disc_key'])) & (mon['ord_key'].eq(r['ord_key']))]
        # Quando houver dia/horário do registro, aplica filtro; se não houver, fica relaxado.
        if r['day_key']:
            mon_match = mon_match[mon_match['day_key'].eq(r['day_key'])]
        if r['hour_key']:
            mon_match = mon_match[mon_match['hour_key'].eq(r['hour_key'])]
        if not cursos.empty:
            mon_match = mon_match[mon_match['curso_key'].isin(set(cursos['curso_key'])) | mon_match['curso_key'].eq('')]
    media_aud = ''
    mediador_raw = ''
    if not mon_match.empty:
        aud = pd.to_numeric(mon_match['AUDIÊNCIA_NUM'], errors='coerce').dropna()
        if not aud.empty:
            media_aud = round(float(aud.mean()), 2)
        meds = [texto_limpo(v) for v in mon_match['MEDIADOR_RAW_LIMPO'].dropna().tolist()]
        meds = [m for m in meds if m and normalizar_texto(m) not in {'NAN','NONE','NSA'}]
        if meds:
            cont = Counter([normalizar_texto(m) for m in meds])
            max_freq = max(cont.values())
            candidatos_keys = {k for k, v in cont.items() if v == max_freq}
            for m in reversed(meds):
                if normalizar_texto(m) in candidatos_keys:
                    mediador_raw = m
                    break
    mediador, chapa2, ch_sem = enriquecer_mediador(mediador_raw, lookup_mediadores)
    return mon_match, media_aud, mediador_raw, mediador, chapa2, ch_sem


def gerar_base(caminho_fluxo, arquivos_mapa, caminho_registro, ano=2026, base_historica=None):
    warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
    registro = carregar_registro(caminho_registro)
    fluxo = carregar_fluxo(caminho_fluxo, ano=ano)

    mapas = {}
    monitoramentos = {}
    metadata_mapas = []
    for arq in arquivos_mapa:
        aba_mapa, mapa = carregar_mapa(arq)
        mod_series = pd.Series(mapa['modulo_num']).dropna()
        modulo = int(mod_series.mode().iloc[0]) if not mod_series.empty else extrair_modulo_num(Path(arq).name)
        mapas[modulo] = mapa
        aba_mon, mon = carregar_monitoramento(arq)
        monitoramentos[modulo] = mon
        metadata_mapas.append({
            'arquivo': Path(arq).name,
            'aba_mapa': aba_mapa,
            'aba_monitoramento': aba_mon,
            'modulo': modulo,
            'linhas_mapa': len(mapa),
            'linhas_monitoramento': len(mon),
        })
    lookup_mediadores = construir_lookup_mediadores(base_historica, monitoramentos)
    modulos_disponiveis = sorted(mapas.keys())
    registro = registro[registro['modulo_num'].isin(modulos_disponiveis)].copy()

    linhas = []
    diagnosticos = []
    for idx, r in registro.iterrows():
        mod = int(r['modulo_num'])
        mapa = mapas[mod]
        fluxo_mod = fluxo[fluxo['SEMESTRE'].eq(mod)]

        cand, rep, tipo_match = selecionar_candidatos_mapa(mapa, r)
        cursos = cand[['curso_key','disc_key','CURSO','CLUSTER']].drop_duplicates(subset=['curso_key','disc_key']) if not cand.empty else pd.DataFrame(columns=['curso_key','disc_key','CURSO','CLUSTER'])
        alunos = 0
        if not cursos.empty:
            alunos_df = cursos[['curso_key','disc_key']].merge(fluxo_mod, on=['curso_key','disc_key'], how='left')
            alunos = int(pd.to_numeric(alunos_df['QTDE_ALUNOS'], errors='coerce').fillna(0).sum())

        mon = monitoramentos.get(mod, pd.DataFrame())
        mon_match, media_aud, mediador_raw, mediador, chapa2, ch_sem = montar_linha_monitoramento(mon, r, cursos, lookup_mediadores)

        professor = texto_limpo(r['PROFESSOR(A)'])
        observacao = texto_limpo(r.get('OBSERVAÇÃO'))
        if normalizar_texto(professor) == 'PROFESSOR RPA' and observacao:
            professor_saida = observacao.upper()
        else:
            professor_saida = professor

        chapa = texto_limpo(r['CHAPA'])
        ch_semanal = r['CH SEMANAL']
        ch_letiva = r['CH LETIVA']
        cargo = texto_limpo(r['CARGO'])
        if normalizar_texto(cargo) == 'RPA' or normalizar_texto(r['PROFESSOR(A)']) == 'PROFESSOR RPA':
            cargo = 'RPA'
            ch_semanal = 0 if pd.isna(ch_semanal) or texto_limpo(ch_semanal) == '' else ch_semanal
            ch_letiva = 0 if pd.isna(ch_letiva) or texto_limpo(ch_letiva) == '' else ch_letiva
            if not chapa:
                chapa = '000000000000'

        # Dia e horário finais: prioriza o registro; quando está vazio, preenche pela aula encontrada no mapa.
        dia_base = r['DIA DA TRANSMISSÃO']
        horario_base = r['HORÁRIO']
        ordem_base = r['ORDEM']
        if (pd.isna(dia_base) or texto_limpo(dia_base) == '') and rep is not None:
            dia_base = rep.get('DIA AULA', '')
        if (pd.isna(horario_base) or texto_limpo(horario_base) == '') and rep is not None:
            horario_base = rep.get('HORÁRIO', '')
        if (pd.isna(ordem_base) or texto_limpo(ordem_base) == '') and rep is not None:
            ordem_base = rep.get('ORDEM', '')

        linha = {
            'CLUSTER': texto_limpo(rep['CLUSTER']) if rep is not None and 'CLUSTER' in rep else '',
            'CURSO': texto_limpo(rep['CURSO']) if rep is not None and 'CURSO' in rep else '',
            'DISCIPLINA': texto_limpo(r['DISCIPLINA (NOME)']),
            'MÓDULO': texto_limpo(r['MÓDULO']),
            'ORDEM': ordem_saida(ordem_base),
            'DIA': dia_saida(dia_base),
            'HORÁRIO': horario_saida(horario_base),
            'QTD_ALUNOS': alunos,
            'MODULAÇÃO': '',
            'PROFESSOR': professor_saida,
            'CHAPA': chapa,
            'CARGO': cargo,
            'CH SEMANAL': ch_semanal if not pd.isna(ch_semanal) else '',
            'CH LETIVA': ch_letiva if not pd.isna(ch_letiva) else '',
            'R$ (RPA)': r.get('R$ (RPA)', ''),
            'MEDIADOR': mediador,
            'CHAPA2': chapa2,
            'CH_SEMANAL': ch_sem,
            'MEDIA_AUDIENCIA': media_aud,
            'CURSOS_SINERGIA': unir_unicos(cursos['CURSO']) if not cursos.empty else '',
            'CLUSTERS_SINERGIA': unir_unicos(cursos['CLUSTER']) if not cursos.empty and 'CLUSTER' in cursos else '',
        }
        linhas.append(linha)

        problemas = []
        if cand.empty:
            problemas.append('Sem correspondência no mapa')
        elif tipo_match != 'exata: disciplina+ordem+dia+horário':
            problemas.append(f'Correspondência {tipo_match}; revisar dia/horário se necessário')
        if alunos == 0:
            problemas.append('QTD_ALUNOS zerada; verificar fluxo ou cursos do mapa')
        if mon is not None and not mon.empty and mon_match.empty:
            problemas.append('Sem correspondência no monitoramento')
        if mediador_raw and not chapa2:
            problemas.append('Mediador sem CHAPA2/CH_SEMANAL; use base histórica/cadastro se precisar enriquecer')
        if problemas:
            diagnosticos.append({
                'linha_registro': idx + 1,
                'MÓDULO': texto_limpo(r['MÓDULO']),
                'DISCIPLINA': texto_limpo(r['DISCIPLINA (NOME)']),
                'ORDEM_REGISTRO': ordem_saida(r['ORDEM']),
                'DIA_REGISTRO': dia_saida(r['DIA DA TRANSMISSÃO']),
                'HORÁRIO_REGISTRO': horario_saida(r['HORÁRIO']),
                'DIA_FINAL': linha['DIA'],
                'HORÁRIO_FINAL': linha['HORÁRIO'],
                'PROFESSOR': professor_saida,
                'TIPO_MATCH_MAPA': tipo_match,
                'PROBLEMA': ' | '.join(problemas),
                'QTD_LINHAS_MAPA': len(cand),
                'QTD_CURSOS_SINERGIA': cursos['curso_key'].nunique() if not cursos.empty else 0,
                'QTD_LINHAS_MONITORAMENTO': len(mon_match) if mon_match is not None else 0,
            })
    base = pd.DataFrame(linhas, columns=COLUNAS_BASE)
    for c in ['QTD_ALUNOS','CH SEMANAL','CH LETIVA','CH_SEMANAL','MEDIA_AUDIENCIA','R$ (RPA)']:
        if c in base.columns:
            convertido = pd.to_numeric(base[c], errors='coerce')
            # Mantém texto original quando a coluna não é predominantemente numérica.
            if convertido.notna().sum() > 0:
                base[c] = convertido.where(convertido.notna(), base[c])
    diag = pd.DataFrame(diagnosticos)
    meta = pd.DataFrame(metadata_mapas)
    return base, diag, meta


def nome_aba_modulo(modulo_texto):
    mod = extrair_modulo_num(modulo_texto)
    ano_match = re.search(r'(20\d{2})', str(modulo_texto))
    ano2 = ano_match.group(1)[2:] if ano_match else ''
    if mod and ano2:
        return f'BASE{mod}{ano2}'[:31]
    if mod:
        return f'BASE{mod}'[:31]
    nome = 'BASE' + re.sub(r'[^A-Za-z0-9]', '', str(modulo_texto))
    return nome[:31]


def resumo_validacao(base, diagnosticos, parametros_processo, base_historica=None):
    linhas = []
    for modulo, df_mod in base.groupby('MÓDULO', dropna=False):
        diag_mod = diagnosticos[diagnosticos['MÓDULO'].eq(modulo)] if not diagnosticos.empty and 'MÓDULO' in diagnosticos else pd.DataFrame()
        linhas.append({
            'MÓDULO': modulo,
            'LINHAS_BASE': len(df_mod),
            'QTD_ALUNOS_TOTAL': pd.to_numeric(df_mod['QTD_ALUNOS'], errors='coerce').fillna(0).sum(),
            'LINHAS_QTD_ALUNOS_ZERO': int((pd.to_numeric(df_mod['QTD_ALUNOS'], errors='coerce').fillna(0) == 0).sum()),
            'LINHAS_DIAGNOSTICO': len(diag_mod),
            'DISCIPLINAS_UNICAS': df_mod['DISCIPLINA'].nunique(dropna=True),
            'PROFESSORES_UNICOS': df_mod['PROFESSOR'].nunique(dropna=True),
        })
    resumo = pd.DataFrame(linhas)
    if base_historica and Path(base_historica).exists():
        try:
            xls = pd.ExcelFile(base_historica)
            hist_linhas = []
            for aba in xls.sheet_names:
                if normalizar_texto(aba).startswith('BASE'):
                    h = pd.read_excel(base_historica, sheet_name=aba)
                    if 'MÓDULO' in h.columns:
                        modulo = texto_limpo(h['MÓDULO'].dropna().iloc[0]) if not h['MÓDULO'].dropna().empty else aba
                    else:
                        modulo = aba
                    hist_linhas.append({
                        'MÓDULO': modulo,
                        'LINHAS_BASE_HISTORICA': len(h),
                        'QTD_ALUNOS_TOTAL_HISTORICA': pd.to_numeric(h.get('QTD_ALUNOS'), errors='coerce').fillna(0).sum() if 'QTD_ALUNOS' in h else np.nan,
                    })
            hist = pd.DataFrame(hist_linhas)
            if not hist.empty and not resumo.empty:
                resumo = resumo.merge(hist, on='MÓDULO', how='left')
                resumo['DIF_LINHAS_VS_HISTORICA'] = resumo['LINHAS_BASE'] - resumo['LINHAS_BASE_HISTORICA']
                resumo['DIF_QTD_ALUNOS_VS_HISTORICA'] = resumo['QTD_ALUNOS_TOTAL'] - resumo['QTD_ALUNOS_TOTAL_HISTORICA']
        except Exception:
            pass
    return resumo


def exportar_base(base, diagnosticos, parametros_processo, saida_xlsx, saida_csv=None, base_historica=None):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    saida_xlsx = Path(saida_xlsx)
    resumo = resumo_validacao(base, diagnosticos, parametros_processo, base_historica=base_historica)

    with pd.ExcelWriter(saida_xlsx, engine='openpyxl') as writer:
        base.to_excel(writer, sheet_name='GERAL', index=False)
        for modulo, df_mod in base.groupby('MÓDULO', dropna=False):
            df_mod.to_excel(writer, sheet_name=nome_aba_modulo(modulo), index=False)
        diagnosticos.to_excel(writer, sheet_name='DIAGNOSTICOS', index=False)
        parametros_processo.to_excel(writer, sheet_name='PARAMETROS_PROCESSO', index=False)
        resumo.to_excel(writer, sheet_name='RESUMO_VALIDACAO', index=False)

    wb = load_workbook(saida_xlsx)
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for ws in wb.worksheets:
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = border
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.border = border
                cell.alignment = Alignment(vertical='top', wrap_text=False)
        for col_idx, col in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in col:
                val = '' if cell.value is None else str(cell.value)
                max_len = max(max_len, len(val))
            width = min(max(max_len + 2, 10), 42)
            ws.column_dimensions[get_column_letter(col_idx)].width = width
        ws.row_dimensions[1].height = 30
    wb.save(saida_xlsx)

    if saida_csv:
        base.to_csv(saida_csv, index=False, encoding='utf-8-sig')
    return resumo


# -----------------------------------------------------------------------------
# Camada de calibração contra base conferida manualmente
# -----------------------------------------------------------------------------

def chave_calibracao(row):
    """Chave usada para confrontar base automática e base conferida.

    Mantém PROFESSOR na chave para preservar co-docência e duplicidades reais da base manual.
    """
    return '|'.join([
        normalizar_texto(row.get('MÓDULO')),
        normalizar_texto(row.get('DISCIPLINA')),
        normalizar_ordem(row.get('ORDEM')),
        normalizar_dia(row.get('DIA')),
        normalizar_horario(row.get('HORÁRIO')),
        normalizar_texto(row.get('PROFESSOR')),
    ])



def decodificar_base_conferida_embutida():
    """Retorna a base conferida 51/52/53 embutida no notebook.

    Por que existe:
    - 51 e 52 já foram conferidos manualmente e não devem ser recalculados pelo fluxo/mapa.
    - 53 foi conferido e complementado com a exceção validada de 80 alunos.
    - Se o notebook for executado apenas com FLUXO + MAPA + REGISTRO, esta base impede
      regressão no número de alunos dos módulos históricos.
    """
    b64 = globals().get('BASE_CONFERIDA_EMBUTIDA_B64', '')
    if not b64:
        return pd.DataFrame(columns=COLUNAS_BASE)
    import base64, gzip, io
    texto_csv = gzip.decompress(base64.b64decode(b64.encode('ascii'))).decode('utf-8')
    df = pd.read_csv(io.StringIO(texto_csv), dtype=str)
    for c in COLUNAS_BASE:
        if c not in df.columns:
            df[c] = np.nan
    return df[COLUNAS_BASE].copy()


def _preparar_df_base_conferida(df, origem='BASE_CONFERIDA'):
    """Padroniza uma base conferida para uso como gabarito de calibração."""
    if df is None or df.empty:
        return pd.DataFrame(columns=COLUNAS_BASE)
    df = df.copy()
    for c in COLUNAS_BASE:
        if c not in df.columns:
            df[c] = np.nan
    df = df[COLUNAS_BASE].copy()
    df['_ABA_ORIGEM'] = origem
    df['_ORIGEM_LINHA'] = origem
    df['_MODULO_NUM'] = df['MÓDULO'].map(extrair_modulo_num)
    df['_CHAVE_CALIBRACAO'] = df.apply(chave_calibracao, axis=1)
    return df


def carregar_base_conferida(caminho_base_conferida):
    """Carrega a base manual/conferida usada como gabarito.

    Ordem de uso:
    1. Se houver arquivo de base conferida na pasta, ele prevalece.
    2. Se não houver arquivo, usa a base conferida embutida no notebook.
    3. Se ambos estiverem ausentes e houver módulos históricos em MODULOS_SAIDA,
       o notebook deve parar antes de exportar para evitar número de alunos incorreto.
    """
    usar_embutida = bool(globals().get('USAR_BASE_CONFERIDA_EMBUTIDA', True))

    if not caminho_base_conferida or not Path(caminho_base_conferida).exists():
        if usar_embutida:
            embutida = decodificar_base_conferida_embutida()
            if not embutida.empty:
                return _preparar_df_base_conferida(embutida, origem='BASE_CONFERIDA_EMBUTIDA')
        return pd.DataFrame(columns=COLUNAS_BASE)

    xls = pd.ExcelFile(caminho_base_conferida)
    abas_origem = []
    if any(normalizar_texto(aba) == 'GERAL' for aba in xls.sheet_names):
        abas_origem = [aba for aba in xls.sheet_names if normalizar_texto(aba) == 'GERAL']
    else:
        abas_origem = [aba for aba in xls.sheet_names if normalizar_texto(aba).startswith('BASE')]

    bases = []
    for aba in abas_origem:
        df = pd.read_excel(caminho_base_conferida, sheet_name=aba)
        if not set(['DISCIPLINA','MÓDULO']).issubset(df.columns):
            continue
        bases.append(_preparar_df_base_conferida(df, origem=f'ARQUIVO:{aba}'))
    return pd.concat(bases, ignore_index=True) if bases else pd.DataFrame(columns=COLUNAS_BASE)

def filtrar_modulos_saida(base, modulos_saida):
    """Mantém na base final apenas os módulos escolhidos para a saída.

    Este filtro corrige a situação em que a automação processa somente um mapa novo,
    mas o Excel final precisa continuar trazendo o histórico conferido. A regra é:
    quem controla as abas finais é MODULOS_SAIDA, não MODULOS_PROCESSAR_AUTO.
    """
    if not modulos_saida:
        return base.copy()
    mods_desejados = {int(m) for m in modulos_saida}
    saida = base[base['MÓDULO'].map(extrair_modulo_num).isin(mods_desejados)].copy()
    mods_encontrados = set(saida['MÓDULO'].map(extrair_modulo_num).dropna().astype(int).tolist())
    faltantes = sorted(mods_desejados - mods_encontrados)
    if faltantes:
        raise ValueError(
            'A base final não contém todos os módulos pedidos em MODULOS_SAIDA. ' +
            'Módulos faltantes: ' + ', '.join(map(str, faltantes)) +
            '. Para consolidar histórico + módulo novo, mantenha uma base conferida na pasta ' +
            'ou inclua o mapa desses módulos em MODULOS_PROCESSAR_AUTO.'
        )
    return saida


def preparar_base_automatica_para_calibracao(base_automatica):
    auto = base_automatica.copy()
    for c in COLUNAS_BASE:
        if c not in auto.columns:
            auto[c] = np.nan
    auto = auto[COLUNAS_BASE].copy()
    auto['_ABA_ORIGEM'] = ''
    auto['_ORIGEM_LINHA'] = 'AUTOMATICA'
    auto['_MODULO_NUM'] = auto['MÓDULO'].map(extrair_modulo_num)
    auto['_CHAVE_CALIBRACAO'] = auto.apply(chave_calibracao, axis=1)
    return auto


def aplicar_calibracao_base_conferida(
    base_automatica,
    caminho_base_conferida=None,
    modulos_fechados_base_conferida=(51, 52),
    modulos_complementar_base_conferida=(53,),
):
    """Preserva a base manual onde ela já está fechada e complementa módulos em andamento.

    Regras padrão desta entrega:
    - 51 e 52: usa exatamente a base conferida, pois já estão fechadas.
    - 53: preserva tudo da base conferida e adiciona somente linhas automáticas ainda ausentes.
    - 54 e próximos: como ainda não existem na base conferida, usa a base automática.
    """
    manual = carregar_base_conferida(caminho_base_conferida)
    auto = preparar_base_automatica_para_calibracao(base_automatica)

    if manual.empty:
        final = auto.copy()
        final['_ORIGEM_LINHA'] = 'AUTOMATICA_SEM_BASE_CONFERIDA'
        incrementos = final.iloc[0:0].copy()
        comparativo = resumo_validacao(final[COLUNAS_BASE], pd.DataFrame(), pd.DataFrame())
        divergencias = pd.DataFrame()
        return final[COLUNAS_BASE], incrementos, comparativo, divergencias

    modulos_fechados = {int(m) for m in modulos_fechados_base_conferida}
    modulos_complementar = {int(m) for m in modulos_complementar_base_conferida}
    modulos = sorted(set(auto['_MODULO_NUM'].dropna().astype(int)) | set(manual['_MODULO_NUM'].dropna().astype(int)))

    saidas = []
    incrementos = []
    for mod in modulos:
        m = manual[manual['_MODULO_NUM'].eq(mod)].copy()
        a = auto[auto['_MODULO_NUM'].eq(mod)].copy()
        if mod in modulos_fechados and not m.empty:
            saidas.append(m)
        elif not m.empty and (mod in modulos_complementar or mod not in modulos_fechados):
            existentes = set(m['_CHAVE_CALIBRACAO'])
            inc = a[~a['_CHAVE_CALIBRACAO'].isin(existentes)].copy()
            inc['_ORIGEM_LINHA'] = 'INCREMENTO_AUTOMATICO'
            saidas.extend([m, inc])
            incrementos.append(inc)
        else:
            a['_ORIGEM_LINHA'] = 'AUTOMATICA_SEM_BASE_CONFERIDA'
            saidas.append(a)

    final_full = pd.concat(saidas, ignore_index=True) if saidas else pd.DataFrame(columns=COLUNAS_BASE)
    incrementos_full = pd.concat(incrementos, ignore_index=True) if incrementos else final_full.iloc[0:0].copy()
    comparativo = montar_comparativo_calibracao(manual, auto, final_full, incrementos_full)
    divergencias = montar_divergencias_auto_manual(manual, auto)
    return final_full[COLUNAS_BASE], incrementos_full, comparativo, divergencias



def preencher_colunas_sinergia(base, base_automatica, ajustes_manuais_sinergia=None):
    """Preenche CURSOS_SINERGIA e CLUSTERS_SINERGIA após a calibração.

    Por que esta etapa existe:
    - Quando 51/52 são preservados pela base conferida/manual, essas duas colunas podem vir vazias.
    - A base automática já leu o mapa e possui a sinergia por aula.
    - O preenchimento usa a chave: módulo + disciplina + ordem + dia + horário.
    - Se o mapa trouxer cluster vazio, o cluster é inferido pelo cadastro curso->cluster já presente na base final.
    """
    ajustes_manuais_sinergia = ajustes_manuais_sinergia or {}
    base = base.copy()
    auto = base_automatica.copy()
    for df in [base, auto]:
        for c in COLUNAS_BASE:
            if c not in df.columns:
                df[c] = np.nan

    def chave_aula(row):
        return (
            normalizar_texto(row.get('MÓDULO')),
            normalizar_texto(row.get('DISCIPLINA')),
            normalizar_ordem(row.get('ORDEM')),
            normalizar_dia(row.get('DIA')),
            normalizar_horario(row.get('HORÁRIO')),
        )

    def normalizar_chave_ajuste(k):
        modulo, disciplina, ordem, dia, horario = k
        return (
            normalizar_texto(modulo), normalizar_texto(disciplina), normalizar_ordem(ordem),
            normalizar_dia(dia), normalizar_horario(horario)
        )

    def split_pipe(valor):
        if pd.isna(valor):
            return []
        partes = [p.strip() for p in str(valor).split('|')]
        return [p for p in partes if p and normalizar_texto(p) not in {'NAN','NONE','NULL'}]

    def unir_listas(valores):
        vistos, saida = set(), []
        for valor in valores:
            for p in split_pipe(valor):
                k = normalizar_texto(p)
                if k and k not in vistos:
                    vistos.add(k)
                    saida.append(p)
        return ' | '.join(saida)

    # Sinergia por aula vinda da base automática/mapa.
    lookup_auto = {}
    tmp_auto = auto.copy()
    tmp_auto['_CHAVE_AULA_SINERGIA'] = tmp_auto.apply(chave_aula, axis=1)
    for chave, grupo in tmp_auto.groupby('_CHAVE_AULA_SINERGIA', dropna=False):
        lookup_auto[chave] = {
            'CURSOS_SINERGIA': unir_listas(grupo['CURSOS_SINERGIA'].tolist()),
            'CLUSTERS_SINERGIA': unir_listas(grupo['CLUSTERS_SINERGIA'].tolist()),
        }

    # Fallback por agrupamento já existente na base final.
    tmp_base = base.copy()
    tmp_base['_CHAVE_AULA_SINERGIA'] = tmp_base.apply(chave_aula, axis=1)
    lookup_grupo_final = {}
    for chave, grupo in tmp_base.groupby('_CHAVE_AULA_SINERGIA', dropna=False):
        lookup_grupo_final[chave] = {
            'CURSOS_SINERGIA': unir_listas(grupo['CURSO'].tolist()),
            'CLUSTERS_SINERGIA': unir_listas(grupo['CLUSTER'].tolist()),
        }

    # Curso -> cluster, com base final + automática. Útil quando CLUSTERS_SINERGIA veio vazio no mapa.
    curso_cluster = {}
    curso_cluster_base = pd.concat([base, auto], ignore_index=True)
    curso_cluster_base['_MOD_KEY'] = curso_cluster_base['MÓDULO'].map(normalizar_texto)
    curso_cluster_base['_CURSO_KEY'] = curso_cluster_base['CURSO'].map(normalizar_texto)
    curso_cluster_base = curso_cluster_base.dropna(subset=['CLUSTER'])
    for (mod_key, curso_key), grupo in curso_cluster_base.groupby(['_MOD_KEY','_CURSO_KEY'], dropna=False):
        clusters = [texto_limpo(x) for x in grupo['CLUSTER'].tolist() if texto_limpo(x)]
        if mod_key and curso_key and clusters:
            curso_cluster[(mod_key, curso_key)] = Counter([normalizar_texto(x) for x in clusters]).most_common(1)[0][0]
            # Recupera a grafia original mais comum.
            originais = [x for x in clusters if normalizar_texto(x) == curso_cluster[(mod_key, curso_key)]]
            curso_cluster[(mod_key, curso_key)] = originais[0]

    ajustes_norm = {}
    for chave, valor in ajustes_manuais_sinergia.items():
        if isinstance(valor, dict):
            cursos = valor.get('CURSOS_SINERGIA', '')
            clusters = valor.get('CLUSTERS_SINERGIA', '')
        else:
            cursos, clusters = valor
        ajustes_norm[normalizar_chave_ajuste(chave)] = {
            'CURSOS_SINERGIA': cursos,
            'CLUSTERS_SINERGIA': clusters,
        }

    def inferir_clusters(row, cursos_sinergia):
        clusters = []
        mod_key = normalizar_texto(row.get('MÓDULO'))
        for curso in split_pipe(cursos_sinergia):
            cluster = curso_cluster.get((mod_key, normalizar_texto(curso)), '')
            if cluster:
                clusters.append(cluster)
        if not clusters and texto_limpo(row.get('CLUSTER')):
            clusters.append(texto_limpo(row.get('CLUSTER')))
        return unir_listas(clusters)

    stats = Counter()
    diagnosticos = []
    for idx, row in base.iterrows():
        chave = chave_aula(row)
        cursos = texto_limpo(row.get('CURSOS_SINERGIA'))
        clusters = texto_limpo(row.get('CLUSTERS_SINERGIA'))

        if not cursos:
            cursos = lookup_auto.get(chave, {}).get('CURSOS_SINERGIA', '')
            if cursos:
                stats['cursos_via_base_automatica'] += 1
        if not clusters:
            clusters = lookup_auto.get(chave, {}).get('CLUSTERS_SINERGIA', '')
            if clusters:
                stats['clusters_via_base_automatica'] += 1

        if not cursos:
            cursos = lookup_grupo_final.get(chave, {}).get('CURSOS_SINERGIA', '')
            if cursos:
                stats['cursos_via_grupo_final'] += 1

        if not clusters:
            clusters = inferir_clusters(row, cursos)
            if clusters:
                stats['clusters_inferidos_por_curso'] += 1

        if chave in ajustes_norm:
            ajuste = ajustes_norm[chave]
            if not cursos and ajuste.get('CURSOS_SINERGIA'):
                cursos = ajuste['CURSOS_SINERGIA']
                stats['cursos_via_ajuste_manual'] += 1
            if not clusters and ajuste.get('CLUSTERS_SINERGIA'):
                clusters = ajuste['CLUSTERS_SINERGIA']
                stats['clusters_via_ajuste_manual'] += 1
            elif not clusters and ajuste.get('CURSOS_SINERGIA'):
                clusters = inferir_clusters(row, ajuste['CURSOS_SINERGIA'])
                if clusters:
                    stats['clusters_inferidos_apos_ajuste'] += 1

        if not cursos and texto_limpo(row.get('CURSO')):
            cursos = texto_limpo(row.get('CURSO'))
            stats['cursos_via_proprio_curso'] += 1
        if not clusters and texto_limpo(row.get('CLUSTER')):
            clusters = texto_limpo(row.get('CLUSTER'))
            stats['clusters_via_proprio_cluster'] += 1

        base.at[idx, 'CURSOS_SINERGIA'] = cursos
        base.at[idx, 'CLUSTERS_SINERGIA'] = clusters

        if not cursos or not clusters:
            diagnosticos.append({
                'LINHA_BASE': idx + 2,
                'MÓDULO': row.get('MÓDULO'),
                'CURSO': row.get('CURSO'),
                'CLUSTER': row.get('CLUSTER'),
                'DISCIPLINA': row.get('DISCIPLINA'),
                'ORDEM': row.get('ORDEM'),
                'DIA': row.get('DIA'),
                'HORÁRIO': row.get('HORÁRIO'),
                'PROFESSOR': row.get('PROFESSOR'),
                'CURSOS_SINERGIA': cursos,
                'CLUSTERS_SINERGIA': clusters,
                'STATUS': 'Ainda sem sinergia completa',
            })

    resumo = pd.DataFrame([{
        'LINHAS_BASE': len(base),
        'CURSOS_SINERGIA_PREENCHIDOS': int(base['CURSOS_SINERGIA'].map(lambda x: bool(texto_limpo(x))).sum()),
        'CLUSTERS_SINERGIA_PREENCHIDOS': int(base['CLUSTERS_SINERGIA'].map(lambda x: bool(texto_limpo(x))).sum()),
        'LINHAS_INCOMPLETAS': len(diagnosticos),
        **dict(stats),
    }])
    return base[COLUNAS_BASE], pd.DataFrame(diagnosticos), resumo



def preencher_cluster_principal(base, base_automatica=None, caminho_base_conferida=None):
    """Preenche a coluna CLUSTER quando ela vier vazia após a calibração.

    Motivo da regra:
    - O mapa do módulo 51 não traz a coluna CLUSTER na aba principal.
    - A base manual/conferida já possui o cluster correto por curso.
    - Para evitar que a aba BASE5126 fique com CLUSTER vazio, esta função monta um
      cadastro curso->cluster a partir da base final, da base automática e da base
      conferida disponível, usando primeiro a chave (módulo, curso) e depois o
      fallback global por curso.

    A função não substitui clusters já preenchidos; ela só completa vazios.
    """
    base = base.copy()
    fontes = []
    for df in [base, base_automatica]:
        if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
            tmp = df.copy()
            for c in ['MÓDULO','CURSO','CLUSTER']:
                if c not in tmp.columns:
                    tmp[c] = np.nan
            fontes.append(tmp[['MÓDULO','CURSO','CLUSTER']])

    if caminho_base_conferida and Path(caminho_base_conferida).exists():
        hist = carregar_base_conferida(caminho_base_conferida)
        if not hist.empty:
            fontes.append(hist[['MÓDULO','CURSO','CLUSTER']])

    if not fontes:
        return base, pd.DataFrame()

    cadastro = pd.concat(fontes, ignore_index=True)
    cadastro['_MOD_KEY'] = cadastro['MÓDULO'].map(normalizar_texto)
    cadastro['_CURSO_KEY'] = cadastro['CURSO'].map(normalizar_texto)
    cadastro['_CLUSTER_LIMPO'] = cadastro['CLUSTER'].map(texto_limpo)
    cadastro = cadastro[(cadastro['_CURSO_KEY'].ne('')) & (cadastro['_CLUSTER_LIMPO'].ne(''))].copy()

    def cluster_mais_frequente(grupo):
        valores = grupo['_CLUSTER_LIMPO'].tolist()
        if not valores:
            return ''
        chave = Counter([normalizar_texto(v) for v in valores]).most_common(1)[0][0]
        for v in valores:
            if normalizar_texto(v) == chave:
                return v
        return valores[0]

    mapa_mod_curso = cadastro.groupby(['_MOD_KEY','_CURSO_KEY'], dropna=False).apply(cluster_mais_frequente).to_dict()
    mapa_curso = cadastro.groupby('_CURSO_KEY', dropna=False).apply(cluster_mais_frequente).to_dict()

    logs = []
    for idx, row in base.iterrows():
        cluster_atual = texto_limpo(row.get('CLUSTER'))
        if cluster_atual:
            continue
        mod_key = normalizar_texto(row.get('MÓDULO'))
        curso_key = normalizar_texto(row.get('CURSO'))
        if not curso_key:
            continue
        cluster_novo = mapa_mod_curso.get((mod_key, curso_key), '') or mapa_curso.get(curso_key, '')
        if cluster_novo:
            base.at[idx, 'CLUSTER'] = cluster_novo
            logs.append({
                'LINHA_BASE': idx + 2,
                'MÓDULO': row.get('MÓDULO'),
                'CURSO': row.get('CURSO'),
                'DISCIPLINA': row.get('DISCIPLINA'),
                'CLUSTER_PREENCHIDO': cluster_novo,
                'STATUS': 'CLUSTER preenchido por cadastro curso->cluster da base conferida/automática',
            })

    return base[COLUNAS_BASE], pd.DataFrame(logs)

def resumo_modulo_calibracao(df, prefixo):
    if df is None or df.empty:
        return pd.DataFrame(columns=['MÓDULO', f'LINHAS_{prefixo}', f'QTD_ALUNOS_{prefixo}'])
    tmp = df.copy()
    tmp['QTD_ALUNOS_NUM'] = pd.to_numeric(tmp['QTD_ALUNOS'], errors='coerce').fillna(0)
    return (tmp.groupby('MÓDULO', dropna=False)
               .agg(**{f'LINHAS_{prefixo}': ('DISCIPLINA','size'),
                       f'QTD_ALUNOS_{prefixo}': ('QTD_ALUNOS_NUM','sum')})
               .reset_index())


def montar_comparativo_calibracao(manual, auto, final, incrementos):
    comp = resumo_modulo_calibracao(manual, 'MANUAL')
    for df, prefix in [(auto, 'AUTO_BRUTA'), (final, 'FINAL'), (incrementos, 'INCREMENTOS')]:
        comp = comp.merge(resumo_modulo_calibracao(df, prefix), on='MÓDULO', how='outer')
    for c in comp.columns:
        if c != 'MÓDULO':
            comp[c] = comp[c].fillna(0)
    comp['DIF_AUTO_BRUTA_VS_MANUAL'] = comp.get('QTD_ALUNOS_AUTO_BRUTA', 0) - comp.get('QTD_ALUNOS_MANUAL', 0)
    comp['DIF_FINAL_VS_MANUAL'] = comp.get('QTD_ALUNOS_FINAL', 0) - comp.get('QTD_ALUNOS_MANUAL', 0)
    comp['DIF_LINHAS_FINAL_VS_MANUAL'] = comp.get('LINHAS_FINAL', 0) - comp.get('LINHAS_MANUAL', 0)
    return comp.sort_values('MÓDULO')


def montar_divergencias_auto_manual(manual, auto):
    if manual.empty or auto.empty:
        return pd.DataFrame()
    cm = Counter(manual['_CHAVE_CALIBRACAO'])
    ca = Counter(auto['_CHAVE_CALIBRACAO'])
    linhas = []
    for chave in sorted(set(cm) | set(ca)):
        dif = ca.get(chave, 0) - cm.get(chave, 0)
        if dif == 0:
            continue
        amostra = auto[auto['_CHAVE_CALIBRACAO'].eq(chave)].head(1)
        if amostra.empty:
            amostra = manual[manual['_CHAVE_CALIBRACAO'].eq(chave)].head(1)
        r = amostra.iloc[0]
        linhas.append({
            'MÓDULO': r.get('MÓDULO'),
            'DISCIPLINA': r.get('DISCIPLINA'),
            'ORDEM': r.get('ORDEM'),
            'DIA': r.get('DIA'),
            'HORÁRIO': r.get('HORÁRIO'),
            'PROFESSOR': r.get('PROFESSOR'),
            'LINHAS_MANUAL': cm.get(chave, 0),
            'LINHAS_AUTO_BRUTA': ca.get(chave, 0),
            'DIF_LINHAS_AUTO_MENOS_MANUAL': dif,
        })
    return pd.DataFrame(linhas)



def aplicar_ajustes_manuais_alunos(base, ajustes_manuais_alunos=None):
    """Aplica ajustes pontuais e auditáveis de QTD_ALUNOS validados no arquivo FLUXO.

    Use esta camada quando a automação não localiza a disciplina pelo caminho padrão
    mapa→fluxo, mas a conferência manual confirma o valor correto no FLUXO.

    A chave do ajuste é: (MÓDULO, DISCIPLINA, ORDEM, DIA, HORÁRIO).
    """
    ajustes_manuais_alunos = ajustes_manuais_alunos or {}
    base = base.copy()
    logs = []

    def chave_normalizada(modulo, disciplina, ordem, dia, horario):
        return (
            normalizar_texto(modulo),
            normalizar_texto(disciplina),
            normalizar_ordem(ordem),
            normalizar_dia(dia),
            normalizar_horario(horario),
        )

    chaves_base = base.apply(
        lambda r: chave_normalizada(r.get('MÓDULO'), r.get('DISCIPLINA'), r.get('ORDEM'), r.get('DIA'), r.get('HORÁRIO')),
        axis=1,
    )

    for chave, ajuste in ajustes_manuais_alunos.items():
        modulo, disciplina, ordem, dia, horario = chave
        chave_norm = chave_normalizada(modulo, disciplina, ordem, dia, horario)
        # IMPORTANTE: chaves_base é uma Series de tuplas.
        # Não use chaves_base.eq(chave_norm), pois o pandas pode interpretar
        # a tupla como uma lista de 5 valores e gerar: ValueError: Lengths must be equal.
        # O map abaixo compara tupla contra tupla linha a linha.
        mask = chaves_base.map(lambda item: item == chave_norm)
        qtd_matches = int(mask.sum())
        valor_novo = ajuste.get('QTD_ALUNOS')

        if qtd_matches == 0:
            logs.append({
                'MÓDULO': modulo,
                'DISCIPLINA': disciplina,
                'ORDEM': ordem,
                'DIA': dia,
                'HORÁRIO': horario,
                'STATUS': 'NÃO APLICADO - chave não encontrada na base',
                'QTD_ALUNOS_ANTES': '',
                'QTD_ALUNOS_DEPOIS': valor_novo,
                'OBSERVAÇÃO': ajuste.get('OBSERVAÇÃO', ''),
            })
            continue

        for idx in base.index[mask]:
            antes = base.at[idx, 'QTD_ALUNOS']
            base.at[idx, 'QTD_ALUNOS'] = valor_novo
            for campo in ['CLUSTER','CURSO','CURSOS_SINERGIA','CLUSTERS_SINERGIA']:
                if campo in ajuste and (pd.isna(base.at[idx, campo]) or texto_limpo(base.at[idx, campo]) == ''):
                    base.at[idx, campo] = ajuste[campo]
            logs.append({
                'MÓDULO': base.at[idx, 'MÓDULO'],
                'DISCIPLINA': base.at[idx, 'DISCIPLINA'],
                'ORDEM': base.at[idx, 'ORDEM'],
                'DIA': base.at[idx, 'DIA'],
                'HORÁRIO': base.at[idx, 'HORÁRIO'],
                'STATUS': 'APLICADO - ajuste validado no FLUXO',
                'QTD_ALUNOS_ANTES': antes,
                'QTD_ALUNOS_DEPOIS': valor_novo,
                'CURSO': base.at[idx, 'CURSO'],
                'CLUSTER': base.at[idx, 'CLUSTER'],
                'OBSERVAÇÃO': ajuste.get('OBSERVAÇÃO', ''),
            })

    return base, pd.DataFrame(logs)


def atualizar_diagnosticos_com_ajustes_alunos(diagnosticos, diagnosticos_ajustes_alunos):
    """Reflete os ajustes manuais de alunos na aba DIAGNOSTICOS."""
    diagnosticos = diagnosticos.copy()
    if diagnosticos_ajustes_alunos is None or diagnosticos_ajustes_alunos.empty:
        return diagnosticos

    for _, aj in diagnosticos_ajustes_alunos.iterrows():
        if not str(aj.get('STATUS','')).startswith('APLICADO'):
            continue
        mask = (
            diagnosticos['MÓDULO'].map(normalizar_texto).eq(normalizar_texto(aj.get('MÓDULO'))) &
            diagnosticos['DISCIPLINA'].map(normalizar_texto).eq(normalizar_texto(aj.get('DISCIPLINA'))) &
            diagnosticos['ORDEM_REGISTRO'].map(normalizar_ordem).eq(normalizar_ordem(aj.get('ORDEM'))) &
            diagnosticos['DIA_FINAL'].map(normalizar_dia).eq(normalizar_dia(aj.get('DIA'))) &
            diagnosticos['HORÁRIO_FINAL'].map(normalizar_horario).eq(normalizar_horario(aj.get('HORÁRIO')))
        )
        texto_ajuste = f"QTD_ALUNOS corrigida de {aj.get('QTD_ALUNOS_ANTES')} para {aj.get('QTD_ALUNOS_DEPOIS')}; ajuste validado no FLUXO."
        diagnosticos.loc[mask, 'TIPO_MATCH_MAPA'] = 'ajuste validado no fluxo: disciplina/curso/módulo'
        diagnosticos.loc[mask, 'PROBLEMA'] = texto_ajuste
        if 'QTD_CURSOS_SINERGIA' in diagnosticos.columns:
            diagnosticos.loc[mask, 'QTD_CURSOS_SINERGIA'] = 1

    return diagnosticos





def remover_duplicatas_inteiras(base, diagnosticos=None):
    """Remove duplicatas integrais da base final e registra a auditoria.

    Duplicata integral significa que todos os campos da linha são iguais aos de
    uma linha anterior da própria base final. A primeira ocorrência é mantida e
    apenas as repetições posteriores são removidas.

    O retorno traz:
    - base_sem_duplicatas: DataFrame final sem duplicatas integrais;
    - diagnosticos_atualizado: DIAGNOSTICOS com o resumo e, quando houver,
      as linhas removidas;
    - duplicatas_removidas: DataFrame específico para auditoria detalhada.
    """
    base = base.copy()
    diagnosticos_base = diagnosticos.copy() if isinstance(diagnosticos, pd.DataFrame) else pd.DataFrame()

    if base.empty:
        resumo = pd.DataFrame([{
            'TIPO_DIAGNOSTICO': 'VERIFICACAO_DUPLICATAS',
            'STATUS': 'OK - base vazia; nenhuma duplicata integral encontrada',
            'DUPLICATAS_REMOVIDAS': 0,
            'OBSERVAÇÃO': 'Verificação executada antes da exportação.'
        }])
        return base, pd.concat([diagnosticos_base, resumo], ignore_index=True, sort=False), pd.DataFrame()

    colunas_comparacao = list(base.columns)
    mask_duplicada = base.duplicated(subset=colunas_comparacao, keep='first')
    qtd_duplicatas = int(mask_duplicada.sum())

    resumo = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'VERIFICACAO_DUPLICATAS',
        'STATUS': 'OK - nenhuma duplicata integral encontrada' if qtd_duplicatas == 0 else 'ATENÇÃO - duplicatas integrais removidas',
        'DUPLICATAS_REMOVIDAS': qtd_duplicatas,
        'OBSERVAÇÃO': 'Linhas inteiras iguais foram avaliadas em todas as colunas da base final. Quando existiram, a primeira ocorrência foi mantida e as repetições removidas.'
    }])

    if qtd_duplicatas == 0:
        diagnosticos_atualizado = pd.concat([diagnosticos_base, resumo], ignore_index=True, sort=False)
        return base.reset_index(drop=True), diagnosticos_atualizado, pd.DataFrame()

    duplicatas = base.loc[mask_duplicada].copy()
    duplicatas.insert(0, 'TIPO_DIAGNOSTICO', 'DUPLICATA_INTEIRA_REMOVIDA')
    duplicatas.insert(1, 'STATUS', 'REMOVIDA - linha inteira duplicada; primeira ocorrência mantida')
    duplicatas.insert(2, 'LINHA_ORIGINAL_BASE', (base.index[mask_duplicada] + 2).astype(int))
    duplicatas.insert(3, 'DUPLICATAS_REMOVIDAS', qtd_duplicatas)
    duplicatas['OBSERVAÇÃO_DUPLICATA'] = 'Removida antes da exportação por ser 100% igual a uma linha anterior da base final.'

    # A própria tabela de duplicatas é anexada ao DIAGNOSTICOS para atender à
    # regra: toda remoção precisa aparecer na aba DIAGNOSTICOS.
    diagnosticos_atualizado = pd.concat([diagnosticos_base, resumo, duplicatas], ignore_index=True, sort=False)
    base_sem_duplicatas = base.loc[~mask_duplicada].copy().reset_index(drop=True)
    return base_sem_duplicatas, diagnosticos_atualizado, duplicatas.reset_index(drop=True)



def verificar_duplicatas_inteiras_sem_remover(base, diagnosticos=None):
    """Verifica duplicatas integrais sem remover linhas da base final.

    Esta versão respeita a decisão de manter a base exatamente como está no
    gabarito/manual. Se houver linhas 100% iguais, elas são reportadas em
    DIAGNOSTICOS e na aba DUPLICATAS_IDENTIFICADAS, mas a base exportada não é
    alterada.
    """
    base = base.copy()
    diagnosticos_base = diagnosticos.copy() if isinstance(diagnosticos, pd.DataFrame) else pd.DataFrame()

    if base.empty:
        resumo = pd.DataFrame([{
            'TIPO_DIAGNOSTICO': 'VERIFICACAO_DUPLICATAS',
            'STATUS': 'OK - base vazia; nenhuma duplicata integral encontrada',
            'DUPLICATAS_IDENTIFICADAS': 0,
            'DUPLICATAS_REMOVIDAS': 0,
            'OBSERVAÇÃO': 'Verificação executada antes da exportação. Nenhuma linha foi removida.'
        }])
        return base, pd.concat([diagnosticos_base, resumo], ignore_index=True, sort=False), pd.DataFrame()

    colunas_comparacao = list(base.columns)
    mask_duplicada = base.duplicated(subset=colunas_comparacao, keep='first')
    qtd_duplicatas = int(mask_duplicada.sum())

    resumo = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'VERIFICACAO_DUPLICATAS',
        'STATUS': 'OK - nenhuma duplicata integral encontrada' if qtd_duplicatas == 0 else 'ATENÇÃO - duplicatas integrais identificadas e mantidas',
        'DUPLICATAS_IDENTIFICADAS': qtd_duplicatas,
        'DUPLICATAS_REMOVIDAS': 0,
        'OBSERVAÇÃO': 'Linhas inteiras iguais foram avaliadas em todas as colunas da base final. A base foi mantida conforme o gabarito/manual; nenhuma duplicata foi removida.'
    }])

    if qtd_duplicatas == 0:
        diagnosticos_atualizado = pd.concat([diagnosticos_base, resumo], ignore_index=True, sort=False)
        return base.reset_index(drop=True), diagnosticos_atualizado, pd.DataFrame()

    duplicatas = base.loc[mask_duplicada, colunas_comparacao].copy()
    duplicatas.insert(0, 'TIPO_DIAGNOSTICO', 'DUPLICATA_INTEGRAL')
    duplicatas.insert(1, 'STATUS', 'IDENTIFICADA - mantida conforme base conferida/manual')
    duplicatas.insert(2, 'LINHA_ORIGINAL_BASE', (base.index[mask_duplicada] + 2).astype(int))
    duplicatas.insert(3, 'DUPLICATAS_IDENTIFICADAS', qtd_duplicatas)
    duplicatas.insert(4, 'DUPLICATAS_REMOVIDAS', 0)
    duplicatas['OBSERVAÇÃO_DUPLICATA'] = 'Duplicata 100% igual a uma linha anterior; mantida por solicitação do usuário.'

    diagnosticos_atualizado = pd.concat([diagnosticos_base, resumo, duplicatas], ignore_index=True, sort=False)
    return base.reset_index(drop=True), diagnosticos_atualizado, duplicatas.reset_index(drop=True)

def validar_totais_calibrados(base, totais_validados=None, linhas_validadas=None, modulos_saida=None, etapa='validação'):
    """Impede exportação com regressão nos módulos já conferidos.

    Esta validação é crítica para o 51: a automação bruta pelo fluxo/mapa pode gerar
    total diferente da base manual. Para módulos fechados, o total final precisa bater
    com o gabarito validado; se não bater, o notebook para antes de criar a planilha.
    """
    totais_validados = totais_validados or {}
    linhas_validadas = linhas_validadas or {}
    modulos_saida = set(int(m) for m in (modulos_saida or []))
    problemas = []
    tmp = base.copy()
    tmp['_MOD_VALIDACAO'] = tmp['MÓDULO'].map(extrair_modulo_num)
    tmp['_QTD_ALUNOS_NUM'] = pd.to_numeric(tmp['QTD_ALUNOS'], errors='coerce').fillna(0)
    for mod, total_esperado in totais_validados.items():
        if modulos_saida and int(mod) not in modulos_saida:
            continue
        df_mod = tmp[tmp['_MOD_VALIDACAO'].eq(int(mod))]
        total_atual = int(df_mod['_QTD_ALUNOS_NUM'].sum())
        linhas_atual = int(len(df_mod))
        linhas_esperadas = linhas_validadas.get(mod)
        if total_atual != int(total_esperado):
            problemas.append(f'Módulo {mod}: QTD_ALUNOS total={total_atual}, esperado={int(total_esperado)}')
        if linhas_esperadas is not None and linhas_atual != int(linhas_esperadas):
            problemas.append(f'Módulo {mod}: linhas={linhas_atual}, esperado={int(linhas_esperadas)}')
    if problemas:
        raise ValueError(
            'Falha na validação de alunos antes da exportação (' + etapa + ').\n' +
            '\n'.join(problemas) +
            '\n\nAção recomendada: mantenha USAR_BASE_CONFERIDA=True e USAR_BASE_CONFERIDA_EMBUTIDA=True, '
            'ou coloque a base conferida/revisada na mesma pasta do notebook.'
        )
    return True

def exportar_base_calibrada(
    base,
    diagnosticos,
    parametros_processo,
    saida_xlsx,
    saida_csv=None,
    base_historica=None,
    incrementos=None,
    comparativo_calibracao=None,
    divergencias_auto_manual=None,
    diagnosticos_sinergia=None,
    resumo_sinergia=None,
    diagnosticos_ajustes_alunos=None,
    diagnosticos_cluster_principal=None,
    diagnosticos_duplicatas=None,
):
    """Exporta a base final em uma única gravação do Excel.

    A versão anterior criava o arquivo e depois reabria em modo append para inserir
    abas extras. No Jupyter Lab isso podia ficar lento. Esta função escreve todas
    as abas de uma vez e só então aplica a formatação.
    """
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    saida_xlsx = Path(saida_xlsx)
    resumo = resumo_validacao(base, diagnosticos, parametros_processo, base_historica=base_historica)

    abas = []
    abas.append(('GERAL', base))
    for modulo, df_mod in base.groupby('MÓDULO', dropna=False):
        abas.append((nome_aba_modulo(modulo), df_mod))
    abas.append(('DIAGNOSTICOS', diagnosticos if diagnosticos is not None else pd.DataFrame()))
    abas.append(('PARAMETROS_PROCESSO', parametros_processo if parametros_processo is not None else pd.DataFrame()))
    abas.append(('RESUMO_VALIDACAO', resumo))

    if incrementos is not None:
        cols = [c for c in COLUNAS_BASE if c in incrementos.columns]
        extras = [c for c in ['_ORIGEM_LINHA'] if c in incrementos.columns]
        abas.append(('INCREMENTOS_AUTOMATICOS', incrementos[cols + extras] if cols or extras else incrementos))
    if comparativo_calibracao is not None:
        abas.append(('COMPARATIVO_CALIBRACAO', comparativo_calibracao))
    if divergencias_auto_manual is not None:
        abas.append(('DIVERGENCIAS_AUTO_MANUAL', divergencias_auto_manual))
    if resumo_sinergia is not None:
        abas.append(('RESUMO_SINERGIA', resumo_sinergia))
    if diagnosticos_sinergia is not None:
        if diagnosticos_sinergia.empty:
            abas.append(('DIAG_SINERGIA', pd.DataFrame([{'STATUS': 'Todas as linhas ficaram com CURSOS_SINERGIA e CLUSTERS_SINERGIA preenchidos.'}])))
        else:
            abas.append(('DIAG_SINERGIA', diagnosticos_sinergia))
    if diagnosticos_ajustes_alunos is not None:
        if diagnosticos_ajustes_alunos.empty:
            abas.append(('AJUSTES_ALUNOS', pd.DataFrame([{'STATUS': 'Nenhum ajuste manual de alunos aplicado.'}])))
        else:
            abas.append(('AJUSTES_ALUNOS', diagnosticos_ajustes_alunos))
    if diagnosticos_cluster_principal is not None:
        if diagnosticos_cluster_principal.empty:
            abas.append(('AJUSTES_CLUSTER', pd.DataFrame([{'STATUS': 'Nenhum CLUSTER principal precisou ser preenchido.'}])))
        else:
            abas.append(('AJUSTES_CLUSTER', diagnosticos_cluster_principal))
    if diagnosticos_duplicatas is not None:
        if diagnosticos_duplicatas.empty:
            abas.append(('DUPLICATAS_IDENTIFICADAS', pd.DataFrame([{'STATUS': 'Nenhuma duplicata integral encontrada. Nenhuma linha removida.'}])))
        else:
            abas.append(('DUPLICATAS_IDENTIFICADAS', diagnosticos_duplicatas))

    nomes_usados = set()
    abas_normalizadas = []
    for nome, df in abas:
        nome_base = str(nome)[:31]
        nome_final = nome_base
        n = 1
        while nome_final in nomes_usados:
            sufixo = f'_{n}'
            nome_final = nome_base[:31-len(sufixo)] + sufixo
            n += 1
        nomes_usados.add(nome_final)
        abas_normalizadas.append((nome_final, df if isinstance(df, pd.DataFrame) else pd.DataFrame(df)))

    with pd.ExcelWriter(saida_xlsx, engine='openpyxl') as writer:
        for nome, df in abas_normalizadas:
            df.to_excel(writer, sheet_name=nome, index=False)

    wb = load_workbook(saida_xlsx)
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for ws in wb.worksheets:
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = border
        # Limita a formatação das abas grandes para não deixar o Jupyter lento.
        max_row_format = min(ws.max_row, 2500)
        for row in ws.iter_rows(min_row=2, max_row=max_row_format):
            for cell in row:
                cell.border = border
                cell.alignment = Alignment(vertical='top', wrap_text=False)
        for col_idx, col in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in col[:min(len(col), 300)]:
                val = '' if cell.value is None else str(cell.value)
                max_len = max(max_len, len(val))
            width = min(max(max_len + 2, 10), 42)
            ws.column_dimensions[get_column_letter(col_idx)].width = width
        ws.row_dimensions[1].height = 30
    wb.save(saida_xlsx)

    if saida_csv:
        base.to_csv(saida_csv, index=False, encoding='utf-8-sig')
    return resumo


In [ ]:
# =============================================================================
# BASE CONFERIDA EMBUTIDA 51/52/53
# =============================================================================
# Esta base é usada como gabarito quando você roda o notebook apenas com
# FLUXO + MAPA + REGISTRO. Ela impede que os módulos já validados sejam
# recalculados e percam alunos, principalmente o módulo 51.
USAR_BASE_CONFERIDA_EMBUTIDA = False  # desativado na versão de portfolio (sem dados embutidos)
#
# NOTA (versão de portfolio): o payload original embutia uma base real de
# alunos codificada em base64/gzip e foi removido por conter dados pessoais.
# Para reproduzir esse fallback com seus próprios dados, gere o payload com:
#   import base64, gzip
#   BASE_CONFERIDA_EMBUTIDA_B64 = base64.b64encode(
#       gzip.compress(caminho_do_csv.read_bytes())
#   ).decode('ascii')
BASE_CONFERIDA_EMBUTIDA_B64 = ''


## 2. Parâmetros

Nesta versão, o notebook separa claramente duas decisões:

- `MODULOS_SAIDA`: módulos que devem aparecer no Excel final.
- `MODULOS_PROCESSAR_AUTO`: módulos que precisam ser lidos dos mapas nesta execução.

Exemplos:

- Regerar o consolidado atual: `MODULOS_SAIDA = [51, 52, 53]` e `MODULOS_PROCESSAR_AUTO = [51, 52, 53]`.
- Gerar o próximo consolidado com 54 mantendo histórico: `MODULOS_SAIDA = [51, 52, 53, 54]` e `MODULOS_PROCESSAR_AUTO = [54]`.
- Gerar somente o 54, sem histórico: `MODULOS_SAIDA = [54]` e `MODULOS_PROCESSAR_AUTO = [54]`.

O erro de sair apenas um módulo geralmente acontece quando `MODULOS_SAIDA` é deixado como `[53]` ou `[54]`. Para o dashboard consolidado, mantenha todos os módulos desejados na saída.


**Importante nesta revisão:** o notebook não sobrescreve mais a base conferida por padrão; ele exporta um arquivo `FINAL_DASHBOARD`. O módulo 53 fica como complementar para preservar a versão manual e adicionar faltantes quando a base local ainda for parcial.

In [ ]:
from pathlib import Path

# =============================================================================
# PARÂMETROS PRINCIPAIS
# =============================================================================
# Pasta onde estão os arquivos de entrada e onde as saídas serão salvas.
# Em uso local, deixe Path.cwd() e coloque os arquivos na mesma pasta do notebook.
PASTA = Path.cwd()

# Bloco específico para o ambiente do ChatGPT. Em uso local, pode ser removido.
if Path('/mnt/data').exists() and (Path('/mnt/data') / 'FLUXO.xlsx').exists():
    PASTA = Path('/mnt/data')

# Ano letivo analisado. Mantenha 2026 enquanto o fluxo/mapa/registro forem desse ciclo.
ANO = 2026

# =============================================================================
# MÓDULOS DE SAÍDA X MÓDULOS PROCESSADOS AUTOMATICAMENTE
# =============================================================================
# MODULOS_SAIDA controla quais módulos aparecem no Excel final e nas abas BASEXXXX.
# Para o consolidado atual, deixe [51, 52, 53].
# Para o próximo consolidado com o módulo 54, use [51, 52, 53, 54].
MODULOS_SAIDA = [51, 52, 53]

# MODULOS_PROCESSAR_AUTO controla quais mapas serão lidos nesta execução.
# Para atualizar o consolidado atual preservando a base conferida/manual, deixe [].
# Para complementar novamente o 53 a partir do mapa, use [53].
# Para reprocessar tudo do zero, use [51, 52, 53] conscientemente.
# Para acrescentar o 54 mantendo 51/52/53 da base conferida, use [54].
MODULOS_PROCESSAR_AUTO = []

# Módulos em que o R$ (RPA) deve ser sincronizado diretamente com o REGISTRO atualizado.
# Mantemos 51/52 como gabarito manual; o 53 recebe esta atualização porque a base manual
# do 53 era parcial e alguns valores de RPA já existiam no REGISTRO, mas estavam vazios na aba BASE5326.
MODULOS_ATUALIZAR_RPA_PELO_REGISTRO = [53]

# Mantido por compatibilidade com versões anteriores do notebook.
MODULOS_PROCESSAR = MODULOS_PROCESSAR_AUTO

# =============================================================================
# POLÍTICA DE CALIBRAÇÃO COM BASE MANUAL
# =============================================================================
# A base manual/conferida funciona como gabarito de qualidade.
# Use True quando houver uma base já validada que deve prevalecer sobre a automação.
USAR_BASE_CONFERIDA = True

# Módulos fechados: preservam 100% do que está na base manual/conferida.
# 51 e 52 ficam fechados porque já foram conferidos manualmente.
MODULOS_FECHADOS_BASE_CONFERIDA = [51, 52]

# Módulos complementares: mantêm a base manual e adicionam linhas novas da automação.
# O 53 fica como complementar por segurança: se a base local ainda for a parcial,
# a automação acrescenta as linhas faltantes; se a base já for a revisada, não duplica.
MODULOS_COMPLEMENTAR_BASE_CONFERIDA = [53]

# Exemplos seguros:
# 1) Regerar 51/52/53 completo:
#    MODULOS_SAIDA = [51, 52, 53]
#    MODULOS_PROCESSAR_AUTO = [51, 52, 53]
#    MODULOS_FECHADOS_BASE_CONFERIDA = [51, 52]
#
# 2) Gerar 51/52/53/54 mantendo histórico e processando só o mapa 54:
#    MODULOS_SAIDA = [51, 52, 53, 54]
#    MODULOS_PROCESSAR_AUTO = [54]
#    MODULOS_FECHADOS_BASE_CONFERIDA = [51, 52]
#    MODULOS_COMPLEMENTAR_BASE_CONFERIDA = [53]
#
# 3) Gerar somente o 54 sem histórico:
#    MODULOS_SAIDA = [54]
#    MODULOS_PROCESSAR_AUTO = [54]
#    USAR_BASE_CONFERIDA = False

# =============================================================================
# LOCALIZAÇÃO DOS ARQUIVOS
# =============================================================================
# Detecção automática por padrão de nome. Se houver ambiguidade, substitua por caminho exato, por exemplo:
# ARQUIVO_FLUXO = PASTA / 'FLUXO_54.xlsx'
ARQUIVO_FLUXO = encontrar_arquivo(PASTA, ['FLUXO*.xlsx'])
ARQUIVO_REGISTRO = encontrar_arquivo(PASTA, ['REGISTRO_PROFESSORES*.xlsx', 'REGISTRO*.xlsx'])
ARQUIVOS_MAPA = localizar_mapas(PASTA, MODULOS_PROCESSAR_AUTO)

# Base manual/conferida usada para preservar módulos já validados.
# A versão revisada é preferida, pois contém o 53 validado com sinergia e ajuste de alunos.
BASES_CONFERIDAS_PRIORIDADE = [
    PASTA / 'Base_Instituicao_515253_revisada_alunos.xlsx',
    PASTA / 'Base_Instituicao_515253_calibrada_sinergia.xlsx',
    PASTA / 'Base_Instituicao_515253.xlsx',
]
ARQUIVO_BASE_HISTORICA = next((p for p in BASES_CONFERIDAS_PRIORIDADE if p.exists()), None)
ARQUIVO_BASE_HISTORICA = ARQUIVO_BASE_HISTORICA if USAR_BASE_CONFERIDA else None

# Validação preventiva: se a saída pede módulos que não serão processados automaticamente,
# eles precisam existir em uma base conferida. Assim o notebook não gera acidentalmente
# apenas o módulo novo ou apenas o último mapa encontrado.
MODULOS_QUE_DEPENDEM_DA_BASE_CONFERIDA = sorted(set(MODULOS_SAIDA) - set(MODULOS_PROCESSAR_AUTO))
if MODULOS_QUE_DEPENDEM_DA_BASE_CONFERIDA and ARQUIVO_BASE_HISTORICA is None:
    raise FileNotFoundError(
        'MODULOS_SAIDA contém módulos que não estão em MODULOS_PROCESSAR_AUTO: ' +
        ', '.join(map(str, MODULOS_QUE_DEPENDEM_DA_BASE_CONFERIDA)) +
        '. Para manter esses módulos na saída, coloque a base conferida na pasta ' +
        'ou processe também os mapas correspondentes.'
    )

# =============================================================================
# NOMES DAS SAÍDAS
# =============================================================================
SUFIXO_MODULOS = ''.join(str(m) for m in MODULOS_SAIDA)
SAIDA_XLSX = PASTA / f'Base_Instituicao_{SUFIXO_MODULOS}_CONFERIDA_FORMATADA_DASHBOARD_RPA53_CORRIGIDO.xlsx'
SAIDA_CSV = PASTA / f'Base_Instituicao_{SUFIXO_MODULOS}_CONFERIDA_FORMATADA_DASHBOARD_RPA53_CORRIGIDO.csv'

# Valores de controle da versão validada. Servem para impedir regressões silenciosas
# nos módulos já conferidos. Para módulos futuros, basta não incluir o módulo aqui
# até que o número seja validado.
TOTAIS_ALUNOS_VALIDADOS = {
    51: 934005,
    52: 875381,
    53: 574210,
}
LINHAS_VALIDADAS = {
    51: 376,
    52: 344,
    53: 398,
}

# Conferência visual dos parâmetros antes de executar o restante.
print('Pasta:', PASTA)
print('Fluxo:', ARQUIVO_FLUXO.name)
print('Registro:', ARQUIVO_REGISTRO.name)
print('Mapas processados automaticamente:', [p.name for p in ARQUIVOS_MAPA])
print('Módulos que sairão no Excel:', MODULOS_SAIDA)
print('Base conferida:', ARQUIVO_BASE_HISTORICA.name if ARQUIVO_BASE_HISTORICA else ('embutida no notebook' if globals().get('USAR_BASE_CONFERIDA_EMBUTIDA', False) else 'não usada'))
print('Módulos fechados pela base conferida:', MODULOS_FECHADOS_BASE_CONFERIDA)
print('Módulos complementados pela automação:', MODULOS_COMPLEMENTAR_BASE_CONFERIDA)
print('Saída Excel:', SAIDA_XLSX.name)
print('Saída CSV:', SAIDA_CSV.name)


## 3. Gerar base automática e aplicar calibração

Nesta etapa o notebook monta a base bruta a partir dos arquivos e depois aplica a política de calibração:

- módulos em `MODULOS_FECHADOS_BASE_CONFERIDA`: usa 100% da base manual;
- módulos em `MODULOS_COMPLEMENTAR_BASE_CONFERIDA`: mantém a base manual e adiciona linhas novas encontradas pela automação;
- módulos fora dessas listas: usa somente a automação.


In [ ]:
# 1) Gera a base automática bruta lendo REGISTRO + MAPA + FLUXO + MONITORAMENTO.
#    Observação: aqui entram apenas os módulos listados em MODULOS_PROCESSAR_AUTO.
#    Quando MODULOS_PROCESSAR_AUTO = [], o notebook preserva a base conferida/manual
#    e não tenta recalcular módulos já validados. Isso é útil para manter 51/52/53
#    exatamente como o gabarito aprovado.
if ARQUIVOS_MAPA:
    base_automatica, diagnosticos, parametros_processo = gerar_base(
        caminho_fluxo=ARQUIVO_FLUXO,
        arquivos_mapa=ARQUIVOS_MAPA,
        caminho_registro=ARQUIVO_REGISTRO,
        ano=ANO,
        base_historica=ARQUIVO_BASE_HISTORICA,
    )
else:
    base_automatica = pd.DataFrame(columns=COLUNAS_BASE)
    diagnosticos = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'PROCESSAMENTO_AUTOMATICO',
        'STATUS': 'PULADO - nenhum módulo automático selecionado',
        'MÓDULO': '',
        'DISCIPLINA': '',
        'ORDEM_REGISTRO': '',
        'DIA_FINAL': '',
        'HORÁRIO_FINAL': '',
        'OBSERVAÇÃO': 'MODULOS_PROCESSAR_AUTO está vazio; a saída será montada a partir da base conferida/manual.'
    }])
    parametros_processo = pd.DataFrame([{
        'arquivo': '',
        'aba_mapa': '',
        'aba_monitoramento': '',
        'modulo': '',
        'linhas_mapa': 0,
        'linhas_monitoramento': 0,
        'observacao': 'Processamento automático pulado por configuração.'
    }])

# 2) Aplica a calibração contra a base conferida.
#    Importante: quando a base manual já foi validada, ela tem prioridade sobre a automação.
base, incrementos_calibracao, comparativo_calibracao, divergencias_auto_manual = aplicar_calibracao_base_conferida(
    base_automatica=base_automatica,
    caminho_base_conferida=ARQUIVO_BASE_HISTORICA,
    modulos_fechados_base_conferida=MODULOS_FECHADOS_BASE_CONFERIDA,
    modulos_complementar_base_conferida=MODULOS_COMPLEMENTAR_BASE_CONFERIDA,
)

# 3) Garante que a planilha final contenha exatamente os módulos definidos em MODULOS_SAIDA.
#    Este é o ponto que evita gerar apenas o módulo processado automaticamente.
base = filtrar_modulos_saida(base, MODULOS_SAIDA)

print('Base automática bruta:', base_automatica.shape)
print('Base final calibrada:', base.shape)
print('Incrementos automáticos adicionados:', len(incrementos_calibracao))
print('\nLinhas finais por módulo:')
print(base['MÓDULO'].value_counts(dropna=False).sort_index().to_string())
print('\nComparativo de calibração:')
display(comparativo_calibracao)
base.head()


## 3.1. Preencher sinergia de curso e cluster

Esta etapa usa a base automática como leitura do mapa para preencher as colunas `CURSOS_SINERGIA` e `CLUSTERS_SINERGIA` também nas linhas preservadas da base conferida/manual.


In [ ]:
# =============================================================================
# AJUSTES AUDITÁVEIS DE SINERGIA
# =============================================================================
# Use este dicionário apenas para exceções confirmadas que não aparecem de forma
# direta na aba principal do mapa. Mantendo a chave explícita, qualquer pessoa
# consegue revisar por que a linha foi preenchida.
#
# Chave: (MÓDULO, DISCIPLINA, ORDEM, DIA, HORÁRIO)
# Valor: cursos e clusters que devem preencher a sinergia.
#
# Para o 54, comece com este dicionário vazio. Inclua exceções somente se a aba
# DIAGNOSTICOS mostrar sinergia ausente e a conferência manual confirmar o valor.
AJUSTES_MANUAIS_SINERGIA = {
    ('53|2026', 'PROJETO EXPERIMENTAL', '1', '3 - Terça', '20:40 às 22:10'): {
        'CURSOS_SINERGIA': 'EGRAD_PPUB',
        'CLUSTERS_SINERGIA': 'COMUNICAÇÃO',
    }
}

# Antes de preencher a sinergia, completa a coluna CLUSTER principal.
# Isso protege especialmente o módulo 51, cujo mapa não possui CLUSTER na aba principal.
base, diagnosticos_cluster_principal = preencher_cluster_principal(
    base=base,
    base_automatica=base_automatica,
    caminho_base_conferida=ARQUIVO_BASE_HISTORICA,
)
print('Clusters principais preenchidos automaticamente:', len(diagnosticos_cluster_principal))

# Preenche CURSOS_SINERGIA e CLUSTERS_SINERGIA na base final calibrada.
base, diagnosticos_sinergia, resumo_sinergia = preencher_colunas_sinergia(
    base=base,
    base_automatica=base_automatica,
    ajustes_manuais_sinergia=AJUSTES_MANUAIS_SINERGIA,
)

print('Preenchimento de sinergia:')
display(resumo_sinergia)
print('Linhas ainda incompletas:', len(diagnosticos_sinergia))
if not diagnosticos_sinergia.empty:
    display(diagnosticos_sinergia.head(20))


# =============================================================================
# AJUSTES AUDITÁVEIS DE ALUNOS
# =============================================================================
# Use este dicionário quando o valor existir no FLUXO, mas não for encontrado pela
# chave exata da automação. Cada ajuste entra na aba DIAGNOSTICOS, evitando que o
# número fique "escondido" no código.
#
# Exemplo atual: PROJETO EXPERIMENTAL foi localizado manualmente no fluxo com 80 alunos.
AJUSTES_MANUAIS_ALUNOS = {
    ('53|2026', 'PROJETO EXPERIMENTAL', '1', '3 - Terça', '20:40 às 22:10'): {
        'QTD_ALUNOS': 80,
        'CURSO': 'EGRAD_PPUB',
        'CLUSTER': 'COMUNICAÇÃO',
        'CURSOS_SINERGIA': 'EGRAD_PPUB',
        'CLUSTERS_SINERGIA': 'COMUNICAÇÃO',
        'OBSERVAÇÃO': 'Valor localizado por conferência manual no FLUXO; exceção mantida até constar de forma exata no mapa.',
    }
}

# Aplica os ajustes e acrescenta a justificativa na tabela de diagnóstico.
base, diagnosticos_ajustes_alunos = aplicar_ajustes_manuais_alunos(base, AJUSTES_MANUAIS_ALUNOS)
diagnosticos = atualizar_diagnosticos_com_ajustes_alunos(diagnosticos, diagnosticos_ajustes_alunos)

print('\nValidação de mediadores:')
base_mediadores = base[base[['MEDIADOR','CHAPA2','CH_SEMANAL','MEDIA_AUDIENCIA']].apply(lambda linha: any(texto_limpo(v) for v in linha), axis=1)].copy()
base_mediadores['CH_SEMANAL_NUM'] = pd.to_numeric(base_mediadores['CH_SEMANAL'], errors='coerce')
print('Linhas com mediador:', len(base_mediadores))
print('Mediadores com CH_SEMANAL acima de 100:', int((base_mediadores['CH_SEMANAL_NUM'] > 100).sum()))


In [ ]:
# =============================================================================
# 3.2. Preservar mediadores e aplicar padrão visual da planilha manual
# =============================================================================
# Esta célula corrige uma regressão observada em versões intermediárias:
# os campos de mediadores podiam ser convertidos de forma incorreta
# (ex.: CH_SEMANAL 40 virava 400, MEDIA_AUDIENCIA 4661 virava 46610).
# A função abaixo usa a base manual/revisada como referência, preserva os valores
# quando encontra a mesma linha e aplica uma normalização de segurança quando não encontra.

from collections import defaultdict, deque
from copy import copy as _copy_style
import math as _math

MEDIADOR_COLUNAS = ['MEDIADOR', 'CHAPA2', 'CH_SEMANAL', 'MEDIA_AUDIENCIA']
CHAVE_MEDIADOR = [
    'CLUSTER', 'CURSO', 'DISCIPLINA', 'MÓDULO', 'ORDEM', 'DIA', 'HORÁRIO',
    'QTD_ALUNOS', 'PROFESSOR', 'CHAPA', 'CARGO', 'CH SEMANAL', 'CH LETIVA'
]


def _texto_chave_mediador(valor):
    """Normaliza valores para comparar linhas entre a base final e as bases de referência."""
    if pd.isna(valor):
        return ''
    if isinstance(valor, float) and valor.is_integer():
        valor = int(valor)
    texto = str(valor).strip()
    if re.fullmatch(r'[-+]?\d+\.0', texto):
        texto = texto[:-2]
    return normalizar_texto(texto)


def _chave_mediador(row):
    return tuple(_texto_chave_mediador(row.get(col, '')) for col in CHAVE_MEDIADOR)


def _numero_seguro(valor):
    if pd.isna(valor) or valor == '':
        return pd.NA
    if isinstance(valor, (int, float, np.integer, np.floating)):
        if pd.isna(valor):
            return pd.NA
        return int(valor) if float(valor).is_integer() else float(valor)
    texto = str(valor).strip().replace('\xa0', '')
    if not texto or texto.lower() in {'nan', 'none', '<na>'}:
        return pd.NA
    if ',' in texto and '.' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')
    try:
        numero = float(texto)
        return int(numero) if numero.is_integer() else numero
    except Exception:
        return valor


def _chapa_texto(valor, completar_zero=False):
    if pd.isna(valor) or valor == '':
        return ''
    texto = str(valor).strip()
    if texto.lower() in {'nan', 'none', '<na>'}:
        return ''
    if re.fullmatch(r'\d+\.0', texto):
        texto = texto[:-2]
    if re.fullmatch(r'\d+', texto) and completar_zero and len(texto) == 11:
        texto = '0' + texto
    return texto


def _normalizar_valores_mediador(valores, corrigir_regressao=False):
    """Normaliza os quatro campos de mediador sem alterar o significado dos dados.

    Quando os valores vêm da base manual/revisada, `corrigir_regressao` fica falso
    para preservar exatamente o número registrado à mão. A correção por divisão de
    10 só é aplicada como fallback quando não há referência confiável.
    """
    out = dict(valores)
    out['MEDIADOR'] = texto_limpo(out.get('MEDIADOR', ''))
    out['CHAPA2'] = _chapa_texto(out.get('CHAPA2', ''), completar_zero=True)

    ch = _numero_seguro(out.get('CH_SEMANAL', ''))
    # Correção de segurança para a regressão 40 -> 400.
    # Só aplica quando não há valor vindo da base manual/revisada.
    if corrigir_regressao and isinstance(ch, (int, float)) and ch > 100:
        ch = ch / 10
    out['CH_SEMANAL'] = int(ch) if isinstance(ch, (int, float)) and float(ch).is_integer() else ch

    audiencia = _numero_seguro(out.get('MEDIA_AUDIENCIA', ''))
    # Correção de segurança para a regressão 4661 -> 46610.
    # Só aplica quando não há valor vindo da base manual/revisada; valores como 1130
    # existentes na base preenchida à mão devem permanecer 1130.
    if corrigir_regressao and isinstance(audiencia, (int, float)) and audiencia >= 1000 and int(audiencia) % 10 == 0:
        audiencia = audiencia / 10
    out['MEDIA_AUDIENCIA'] = int(audiencia) if isinstance(audiencia, (int, float)) and float(audiencia).is_integer() else audiencia
    return out


def _ler_referencia_mediadores(caminho):
    """Lê uma base de referência. Se existir GERAL, usa só GERAL; caso contrário, usa abas BASEXXXX."""
    caminho = Path(caminho)
    if not caminho.exists():
        return pd.DataFrame(columns=COLUNAS_BASE)
    try:
        abas = pd.read_excel(caminho, sheet_name=None, dtype=str)
    except Exception:
        return pd.DataFrame(columns=COLUNAS_BASE)
    if 'GERAL' in abas:
        df = abas['GERAL'].copy()
    else:
        partes = []
        for nome, aba in abas.items():
            if str(nome).upper().startswith('BASE') and set(MEDIADOR_COLUNAS).issubset(aba.columns):
                partes.append(aba.copy())
        df = pd.concat(partes, ignore_index=True, sort=False) if partes else pd.DataFrame(columns=COLUNAS_BASE)
    for col in COLUNAS_BASE:
        if col not in df.columns:
            df[col] = pd.NA
    return df[COLUNAS_BASE].copy()


def preservar_mediadores_e_formatacao_dados(base, diagnosticos=None):
    """Preserva os valores de mediadores usando as bases de referência disponíveis.

    Prioridade das referências:
    1. Base manual preenchida à mão, pois preserva exatamente `CHAPA2` com zero à esquerda.
    2. Base revisada anterior, pois contém os incrementos do 53 já validados.
    3. Valor atual da linha com normalização de segurança.
    """
    base = base.copy()
    diagnosticos = diagnosticos.copy() if isinstance(diagnosticos, pd.DataFrame) else pd.DataFrame()

    referencias = [
        PASTA / 'Base_Instituicao_515253.xlsx',
        PASTA / 'Base_Instituicao_515253_revisada_alunos.xlsx',
        PASTA / 'Base_Instituicao_515253_calibrada_sinergia.xlsx',
    ]
    filas_por_referencia = []
    nomes_referencia = []
    for ref in referencias:
        df_ref = _ler_referencia_mediadores(ref)
        if df_ref.empty:
            continue
        filas = defaultdict(deque)
        for _, row in df_ref.iterrows():
            filas[_chave_mediador(row)].append({col: row.get(col, pd.NA) for col in MEDIADOR_COLUNAS})
        filas_por_referencia.append(filas)
        nomes_referencia.append(Path(ref).name)

    detalhes = []
    contagem_fontes = Counter()
    for idx, row in base.iterrows():
        chave = _chave_mediador(row)
        fonte = 'normalizacao_segura'
        valores = None
        for nome_ref, filas in zip(nomes_referencia, filas_por_referencia):
            if filas.get(chave):
                valores = filas[chave].popleft()
                fonte = nome_ref
                break
        if valores is None:
            valores = {col: row.get(col, pd.NA) for col in MEDIADOR_COLUNAS}

        antes = {col: row.get(col, pd.NA) for col in MEDIADOR_COLUNAS}
        depois = _normalizar_valores_mediador(valores, corrigir_regressao=(fonte == 'normalizacao_segura'))
        for col in MEDIADOR_COLUNAS:
            base.at[idx, col] = depois.get(col, pd.NA)

        # Mantém CHAPA e CHAPA2 como texto para não perder zero à esquerda no Excel.
        base.at[idx, 'CHAPA'] = _chapa_texto(base.at[idx, 'CHAPA'], completar_zero=False)
        base.at[idx, 'CHAPA2'] = _chapa_texto(base.at[idx, 'CHAPA2'], completar_zero=True)

        mudou = any(texto_limpo(antes.get(c, '')) != texto_limpo(depois.get(c, '')) for c in MEDIADOR_COLUNAS)
        if mudou:
            detalhes.append({
                'TIPO_DIAGNOSTICO': 'VALIDACAO_MEDIADORES_FORMATACAO',
                'STATUS': 'AJUSTADO - mediadores preservados/normalizados',
                'LINHA_BASE_FINAL': int(idx) + 2,
                'MÓDULO': base.at[idx, 'MÓDULO'],
                'CURSO': base.at[idx, 'CURSO'],
                'DISCIPLINA': base.at[idx, 'DISCIPLINA'],
                'PROFESSOR': base.at[idx, 'PROFESSOR'],
                'FONTE_MEDIADOR': fonte,
                'MEDIADOR_ANTES': antes.get('MEDIADOR', ''),
                'MEDIADOR_DEPOIS': depois.get('MEDIADOR', ''),
                'CHAPA2_ANTES': antes.get('CHAPA2', ''),
                'CHAPA2_DEPOIS': depois.get('CHAPA2', ''),
                'CH_SEMANAL_ANTES': antes.get('CH_SEMANAL', ''),
                'CH_SEMANAL_DEPOIS': depois.get('CH_SEMANAL', ''),
                'MEDIA_AUDIENCIA_ANTES': antes.get('MEDIA_AUDIENCIA', ''),
                'MEDIA_AUDIENCIA_DEPOIS': depois.get('MEDIA_AUDIENCIA', ''),
                'OBSERVAÇÃO': 'Preservação/correção aplicada antes da exportação.'
            })
        contagem_fontes[fonte] += 1

    resumo = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'VALIDACAO_MEDIADORES_FORMATACAO',
        'STATUS': 'OK - mediadores preservados e normalizados antes da exportação',
        'LINHAS_AVALIADAS': len(base),
        'LINHAS_MEDIADOR_AJUSTADAS': len(detalhes),
        'FONTES_USADAS': dict(contagem_fontes),
        'OBSERVAÇÃO': 'CHAPA/CHAPA2 foram mantidas como texto; CH_SEMANAL e MEDIA_AUDIENCIA foram preservados da base manual/revisada quando possível.'
    }])
    diag_mediadores = pd.concat([resumo, pd.DataFrame(detalhes)], ignore_index=True, sort=False)
    diagnosticos = pd.concat([diagnosticos, diag_mediadores], ignore_index=True, sort=False)
    return base, diagnosticos, diag_mediadores


def aplicar_formatacao_manual_planilha(saida_xlsx, diagnosticos_mediadores=None):
    """Aplica o padrão visual verde da planilha manual nas abas de dados.

    A formatação é aplicada depois da exportação para manter a escrita rápida e
    garantir que a saída final fique no mesmo padrão visual da base preenchida à mão.
    """
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    caminho = Path(saida_xlsx)
    wb = load_workbook(caminho)

    modelo = PASTA / 'Base_Instituicao_515253.xlsx'
    larguras = {}
    if modelo.exists():
        try:
            wb_modelo = load_workbook(modelo, read_only=False)
            ws_modelo = wb_modelo['BASE5126']
            for idx in range(1, len(COLUNAS_BASE) + 1):
                letra = get_column_letter(idx)
                larguras[letra] = ws_modelo.column_dimensions[letra].width
        except Exception:
            larguras = {}

    header_fill = PatternFill('solid', fgColor='70AD47')
    odd_fill = PatternFill('solid', fgColor='E2F0D9')
    even_fill = PatternFill('solid', fgColor='FFFFFF')
    header_font = Font(color='FFFFFF', bold=True, name='Calibri', size=8)
    body_font = Font(color='000000', name='Calibri', size=8)
    thin = Side(style='thin', color='A9D18E')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    abas_dados = ['GERAL'] + [nome_aba_modulo(f'{m}|{ANO}') for m in MODULOS_SAIDA]
    for nome in abas_dados:
        if nome not in wb.sheetnames:
            continue
        ws = wb[nome]
        headers = [ws.cell(1, col).value for col in range(1, ws.max_column + 1)]
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = border
        for row_idx in range(2, ws.max_row + 1):
            fill = odd_fill if row_idx % 2 == 0 else even_fill
            for col_idx in range(1, ws.max_column + 1):
                cell = ws.cell(row_idx, col_idx)
                col_name = headers[col_idx - 1] if col_idx - 1 < len(headers) else ''
                cell.fill = fill
                cell.font = body_font
                cell.border = border
                if col_name in {'QTD_ALUNOS', 'CH SEMANAL', 'CH LETIVA', 'CH_SEMANAL', 'MEDIA_AUDIENCIA'}:
                    cell.number_format = '0'
                    cell.alignment = Alignment(horizontal='right', vertical='center', wrap_text=False)
                elif col_name == 'R$ (RPA)':
                    cell.number_format = '0.##'
                    cell.alignment = Alignment(horizontal='right', vertical='center', wrap_text=False)
                elif col_name in {'CHAPA', 'CHAPA2', 'MÓDULO'}:
                    cell.number_format = '@'
                    if cell.value is not None:
                        cell.value = str(cell.value)
                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=False)
                else:
                    cell.alignment = Alignment(horizontal='left', vertical='center', wrap_text=False)
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = f'A1:{get_column_letter(ws.max_column)}{ws.max_row}'
        for idx in range(1, min(ws.max_column, len(COLUNAS_BASE)) + 1):
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].width = larguras.get(letra, ws.column_dimensions[letra].width or 12)
        ws.row_dimensions[1].height = 15

    # Aba auxiliar detalhada dos mediadores.
    if isinstance(diagnosticos_mediadores, pd.DataFrame):
        if 'CONTROLE_MEDIADORES' in wb.sheetnames:
            del wb['CONTROLE_MEDIADORES']
        ws = wb.create_sheet('CONTROLE_MEDIADORES')
        def _valor_excel(valor):
            if pd.isna(valor):
                return ''
            if isinstance(valor, (dict, list, tuple, set)):
                return str(valor)
            return valor
        if diagnosticos_mediadores.empty:
            ws.append(['STATUS'])
            ws.append(['Nenhum ajuste de mediador foi necessário.'])
        else:
            cols = list(diagnosticos_mediadores.columns)
            ws.append(cols)
            for _, row in diagnosticos_mediadores.iterrows():
                ws.append([_valor_excel(row.get(col, '')) for col in cols])

    # Formatação simples verde nas abas auxiliares.
    for ws in wb.worksheets:
        if ws.title in abas_dados or ws.max_row < 1:
            continue
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = Font(color='FFFFFF', bold=True, name='Calibri', size=9)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = border
        ws.freeze_panes = 'A2'
        try:
            ws.auto_filter.ref = ws.dimensions
        except Exception:
            pass
        for col_idx in range(1, ws.max_column + 1):
            max_len = max(len(str(ws.cell(row, col_idx).value or '')) for row in range(1, min(ws.max_row, 200) + 1))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 48)

    wb.save(caminho)
    return caminho

In [ ]:

# =============================================================================
# 3.3. Preservar e validar R$ (RPA)
# =============================================================================
# Esta célula preserva os valores de RPA da base manual/gabarito.
# Ela corrige a regressão em que o valor de RPA do módulo 51 era exportado
# multiplicado por 10 (ex.: 1110 virava 11100).

RPA_COLUNA = 'R$ (RPA)'
CHAVE_RPA = [
    'CLUSTER', 'CURSO', 'DISCIPLINA', 'MÓDULO', 'ORDEM', 'DIA', 'HORÁRIO',
    'QTD_ALUNOS', 'PROFESSOR', 'CHAPA', 'CARGO', 'CH SEMANAL', 'CH LETIVA'
]


def _texto_chave_rpa(valor):
    """Normaliza valores para comparar linhas entre a base final e as referências."""
    if pd.isna(valor):
        return ''
    if isinstance(valor, float) and valor.is_integer():
        valor = int(valor)
    texto = str(valor).strip()
    if re.fullmatch(r'[-+]?\d+\.0', texto):
        texto = texto[:-2]
    return normalizar_texto(texto)


def _chave_rpa(row):
    return tuple(_texto_chave_rpa(row.get(col, '')) for col in CHAVE_RPA)


def _numero_rpa_seguro(valor):
    """Converte RPA para número sem multiplicar ou perder centavos."""
    if pd.isna(valor) or valor == '':
        return pd.NA
    if isinstance(valor, (int, float, np.integer, np.floating)):
        if pd.isna(valor):
            return pd.NA
        numero = float(valor)
        return int(numero) if numero.is_integer() else round(numero, 2)
    texto = str(valor).strip().replace('\xa0', '').replace('R$', '').strip()
    if not texto or texto.lower() in {'nan', 'none', '<na>'}:
        return pd.NA
    texto = texto.replace(' ', '')
    if ',' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    try:
        numero = float(texto)
        return int(numero) if numero.is_integer() else round(numero, 2)
    except Exception:
        return valor


def _ler_referencia_rpa(caminho):
    """Lê referências para preservar valores conferidos de RPA."""
    caminho = Path(caminho)
    if not caminho.exists():
        return pd.DataFrame(columns=COLUNAS_BASE)
    try:
        abas = pd.read_excel(caminho, sheet_name=None, dtype=str)
    except Exception:
        return pd.DataFrame(columns=COLUNAS_BASE)
    if 'GERAL' in abas:
        df = abas['GERAL'].copy()
    else:
        partes = []
        for nome, aba in abas.items():
            if str(nome).upper().startswith('BASE') and RPA_COLUNA in aba.columns:
                partes.append(aba.copy())
        df = pd.concat(partes, ignore_index=True, sort=False) if partes else pd.DataFrame(columns=COLUNAS_BASE)
    for col in COLUNAS_BASE:
        if col not in df.columns:
            df[col] = pd.NA
    return df[COLUNAS_BASE].copy()


def preservar_rpa_e_formatacao_dados(base, diagnosticos=None):
    """Preserva `R$ (RPA)` da base manual e normaliza linhas novas.

    Prioridade:
    1. `Base_Instituicao_515253.xlsx`, que é a base preenchida à mão.
    2. Bases revisadas anteriores, úteis para complementos já validados.
    3. Valor atual da linha, convertido de forma segura.
    """
    base = base.copy()
    diagnosticos = diagnosticos.copy() if isinstance(diagnosticos, pd.DataFrame) else pd.DataFrame()

    referencias = [
        PASTA / 'Base_Instituicao_515253.xlsx',
        PASTA / 'Base_Instituicao_515253_revisada_alunos.xlsx',
        PASTA / 'Base_Instituicao_515253_calibrada_sinergia.xlsx',
    ]

    filas_por_referencia = []
    nomes_referencia = []
    for ref in referencias:
        df_ref = _ler_referencia_rpa(ref)
        if df_ref.empty or RPA_COLUNA not in df_ref.columns:
            continue
        filas = defaultdict(deque)
        for _, row in df_ref.iterrows():
            valor_ref = _numero_rpa_seguro(row.get(RPA_COLUNA, pd.NA))
            if pd.isna(valor_ref):
                continue
            filas[_chave_rpa(row)].append(valor_ref)
        filas_por_referencia.append(filas)
        nomes_referencia.append(Path(ref).name)

    detalhes = []
    total_rpa_por_modulo = Counter()
    for idx, row in base.iterrows():
        antes = row.get(RPA_COLUNA, pd.NA)
        valor_atual = _numero_rpa_seguro(antes)
        valor_final = valor_atual
        fonte = 'valor_atual_normalizado'

        chave = _chave_rpa(row)
        for nome_ref, filas in zip(nomes_referencia, filas_por_referencia):
            if filas.get(chave):
                valor_final = filas[chave].popleft()
                fonte = nome_ref
                break

        # Segurança extra: se ainda restar um valor numérico suspeito do módulo 51,
        # e ele vier multiplicado por 10, corrige e registra. A prioridade continua
        # sendo a base manual; essa linha só protege execuções parciais sem referência.
        if fonte == 'valor_atual_normalizado' and _texto_chave_rpa(row.get('MÓDULO')) == '51|2026':
            if isinstance(valor_final, (int, float)) and valor_final > 0 and abs(valor_final) >= 9000 and abs(valor_final % 10) < 1e-9:
                valor_final = valor_final / 10
                valor_final = int(valor_final) if float(valor_final).is_integer() else round(valor_final, 2)
                fonte = 'normalizacao_segura_modulo_51'

        if pd.isna(valor_final):
            base.at[idx, RPA_COLUNA] = pd.NA
        else:
            base.at[idx, RPA_COLUNA] = valor_final
            try:
                total_rpa_por_modulo[str(row.get('MÓDULO'))] += float(valor_final)
            except Exception:
                pass

        antes_num = _numero_rpa_seguro(antes)
        mudou = False
        try:
            mudou = (pd.isna(antes_num) and not pd.isna(valor_final)) or (not pd.isna(antes_num) and not pd.isna(valor_final) and round(float(antes_num), 2) != round(float(valor_final), 2))
        except Exception:
            mudou = texto_limpo(antes) != texto_limpo(valor_final)

        if mudou:
            detalhes.append({
                'TIPO_DIAGNOSTICO': 'VALIDACAO_RPA_FORMATACAO',
                'STATUS': 'AJUSTADO - RPA preservado/normalizado',
                'LINHA_BASE_FINAL': int(idx) + 2,
                'MÓDULO': row.get('MÓDULO', ''),
                'CURSO': row.get('CURSO', ''),
                'DISCIPLINA': row.get('DISCIPLINA', ''),
                'PROFESSOR': row.get('PROFESSOR', ''),
                'RPA_ANTES': antes,
                'RPA_DEPOIS': valor_final,
                'FONTE_RPA': fonte,
                'OBSERVAÇÃO': 'Valor de RPA validado antes da exportação; evita multiplicação por 10.'
            })

    resumo = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'VALIDACAO_RPA_FORMATACAO',
        'STATUS': 'OK - RPA preservado e validado antes da exportação',
        'LINHAS_AVALIADAS': len(base),
        'LINHAS_RPA_AJUSTADAS': len(detalhes),
        'TOTAL_RPA_POR_MODULO': dict(total_rpa_por_modulo),
        'OBSERVAÇÃO': 'A coluna R$ (RPA) é mantida numérica; módulo 51 usa a base manual como gabarito.'
    }])
    diag_rpa = pd.concat([resumo, pd.DataFrame(detalhes)], ignore_index=True, sort=False)
    diagnosticos = pd.concat([diagnosticos, diag_rpa], ignore_index=True, sort=False)
    return base, diagnosticos, diag_rpa


# =============================================================================
# 3.3.1. Sincronizar R$ (RPA) de módulos complementares com o REGISTRO atualizado
# =============================================================================
# Em módulos complementares, como o 53, algumas linhas já existiam na base manual parcial
# e por isso tiveram A:S preservadas antes do registro mais atual receber os valores de RPA.
# Esta função corrige esse caso sem mexer no gabarito dos módulos fechados 51/52.

CHAVE_RPA_REGISTRO = ['MÓDULO', 'DISCIPLINA', 'ORDEM', 'DIA', 'HORÁRIO']


def _chave_rpa_registro_base(row):
    return (
        _texto_chave_rpa(row.get('MÓDULO', '')),
        _texto_chave_rpa(row.get('DISCIPLINA', '')),
        normalizar_ordem(row.get('ORDEM', '')),
        normalizar_dia(row.get('DIA', '')),
        normalizar_horario(row.get('HORÁRIO', '')),
    )


def _chave_rpa_registro_linha(row):
    return (
        _texto_chave_rpa(row.get('MÓDULO', '')),
        _texto_chave_rpa(row.get('DISCIPLINA (NOME)', '')),
        normalizar_ordem(row.get('ORDEM', '')),
        normalizar_dia(row.get('DIA DA TRANSMISSÃO', '')),
        normalizar_horario(row.get('HORÁRIO', '')),
    )


def atualizar_rpa_por_registro_modulos(base, diagnosticos=None, modulos=None):
    """Atualiza `R$ (RPA)` por REGISTRO em módulos complementares.

    Uso principal nesta entrega: módulo 53.
    A chave é `MÓDULO + DISCIPLINA + ORDEM + DIA + HORÁRIO`, a mesma estrutura
    usada para localizar cursos no mapa. O valor é mantido numérico para Power BI.
    """
    base = base.copy()
    diagnosticos = diagnosticos.copy() if isinstance(diagnosticos, pd.DataFrame) else pd.DataFrame()
    modulos = set(modulos or [])
    if not modulos:
        diag = pd.DataFrame([{
            'TIPO_DIAGNOSTICO': 'VALIDACAO_RPA_REGISTRO',
            'STATUS': 'PULADO - nenhum módulo configurado para sincronizar RPA pelo REGISTRO',
            'OBSERVAÇÃO': 'Configure MODULOS_ATUALIZAR_RPA_PELO_REGISTRO quando um módulo complementar precisar receber valores de RPA do registro atualizado.'
        }])
        return base, pd.concat([diagnosticos, diag], ignore_index=True, sort=False), diag

    try:
        registro = carregar_registro(ARQUIVO_REGISTRO)
    except Exception as exc:
        diag = pd.DataFrame([{
            'TIPO_DIAGNOSTICO': 'VALIDACAO_RPA_REGISTRO',
            'STATUS': 'ERRO - não foi possível ler REGISTRO para sincronizar RPA',
            'OBSERVAÇÃO': str(exc)
        }])
        return base, pd.concat([diagnosticos, diag], ignore_index=True, sort=False), diag

    registro = registro[registro['modulo_num'].isin(modulos)].copy()
    registro['RPA_NUM'] = registro[RPA_COLUNA].map(_numero_rpa_seguro)
    registro = registro[registro['RPA_NUM'].map(lambda x: not pd.isna(x))].copy()

    rpa_por_chave = {}
    for _, row in registro.iterrows():
        rpa_por_chave[_chave_rpa_registro_linha(row)] = row['RPA_NUM']

    detalhes = []
    for idx, row in base.iterrows():
        mod = extrair_modulo_num(row.get('MÓDULO', ''))
        if mod not in modulos:
            continue
        chave = _chave_rpa_registro_base(row)
        if chave not in rpa_por_chave:
            continue
        antes = _numero_rpa_seguro(row.get(RPA_COLUNA, pd.NA))
        depois = rpa_por_chave[chave]
        if pd.isna(depois):
            continue
        base.at[idx, RPA_COLUNA] = depois
        try:
            mudou = (pd.isna(antes) and not pd.isna(depois)) or (not pd.isna(antes) and round(float(antes), 2) != round(float(depois), 2))
        except Exception:
            mudou = texto_limpo(antes) != texto_limpo(depois)
        if mudou:
            detalhes.append({
                'TIPO_DIAGNOSTICO': 'VALIDACAO_RPA_REGISTRO',
                'STATUS': 'AJUSTADO - RPA sincronizado com REGISTRO atualizado',
                'LINHA_BASE_FINAL': int(idx) + 2,
                'MÓDULO': row.get('MÓDULO', ''),
                'CURSO': row.get('CURSO', ''),
                'DISCIPLINA': row.get('DISCIPLINA', ''),
                'ORDEM_REGISTRO': row.get('ORDEM', ''),
                'DIA_FINAL': row.get('DIA', ''),
                'HORÁRIO_FINAL': row.get('HORÁRIO', ''),
                'PROFESSOR': row.get('PROFESSOR', ''),
                'RPA_ANTES': antes,
                'RPA_DEPOIS': depois,
                'OBSERVAÇÃO': 'Módulo complementar: valor de RPA atualizado pela chave DISCIPLINA+ORDEM+DIA+HORÁRIO do REGISTRO.'
            })

    totais = []
    for mod in sorted(modulos):
        mask_mod = base['MÓDULO'].map(extrair_modulo_num).eq(mod)
        serie = base.loc[mask_mod, RPA_COLUNA].map(_numero_rpa_seguro)
        serie = serie[serie.map(lambda x: not pd.isna(x))]
        totais.append({
            'MÓDULO': f'{mod}|{ANO}',
            'RPA_BASE_QTD': int(len(serie)),
            'TOTAL_RPA_BASE': round(float(pd.to_numeric(serie, errors='coerce').fillna(0).sum()), 2),
            'RPA_REGISTRO_QTD': int(len([k for k in rpa_por_chave if k[0] == f'{mod}|{ANO}'])),
            'TOTAL_RPA_REGISTRO': round(float(sum(float(v) for k, v in rpa_por_chave.items() if k[0] == f'{mod}|{ANO}')), 2),
        })

    resumo = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'VALIDACAO_RPA_REGISTRO',
        'STATUS': 'OK - RPA sincronizado com REGISTRO atualizado em módulos complementares',
        'MODULOS_ATUALIZADOS': sorted(modulos),
        'LINHAS_RPA_AJUSTADAS': len(detalhes),
        'RESUMO_RPA': totais,
        'OBSERVAÇÃO': '51/52 permanecem como gabarito manual; 53 recebe RPA do registro atualizado para evitar valores inferiores no Power BI.'
    }])
    diag = pd.concat([resumo, pd.DataFrame(detalhes)], ignore_index=True, sort=False)
    diagnosticos = pd.concat([diagnosticos, diag], ignore_index=True, sort=False)
    return base, diagnosticos, diag


In [ ]:

# =============================================================================
# 3.4. Preservar dados manuais conferidos A:S
# =============================================================================
# A base preenchida à mão é o gabarito para os módulos já conferidos.
# Portanto, quando uma linha já existe na base manual, preservamos as colunas A:S
# exatamente como foram registradas, incluindo CHAPA com zero à esquerda,
# R$ (RPA), mediadores e audiências. A automação continua preenchendo/enriquecendo
# CURSOS_SINERGIA e CLUSTERS_SINERGIA nas colunas T:U.

COLUNAS_PRESERVAR_MANUAL = [c for c in COLUNAS_BASE[:19]]
CHAVE_PRESERVACAO_MANUAL = [
    'CLUSTER', 'CURSO', 'DISCIPLINA', 'MÓDULO', 'ORDEM', 'DIA', 'HORÁRIO',
    'QTD_ALUNOS', 'PROFESSOR'
]


def _texto_chave_preservacao(valor):
    if pd.isna(valor):
        return ''
    if isinstance(valor, float) and valor.is_integer():
        valor = int(valor)
    texto = str(valor).strip()
    if re.fullmatch(r'[-+]?\d+\.0', texto):
        texto = texto[:-2]
    return normalizar_texto(texto)


def _chave_preservacao_manual(row):
    return tuple(_texto_chave_preservacao(row.get(col, '')) for col in CHAVE_PRESERVACAO_MANUAL)


def _ler_base_manual_preservacao(caminho):
    caminho = Path(caminho)
    if not caminho.exists():
        return pd.DataFrame(columns=COLUNAS_BASE)
    try:
        abas = pd.read_excel(caminho, sheet_name=None, dtype=str)
    except Exception:
        return pd.DataFrame(columns=COLUNAS_BASE)
    partes = []
    for nome, aba in abas.items():
        nome_norm = normalizar_texto(nome)
        if nome_norm == 'GERAL' or nome_norm.startswith('BASE'):
            if set(CHAVE_PRESERVACAO_MANUAL).issubset(aba.columns):
                partes.append(aba.copy())
    if not partes:
        return pd.DataFrame(columns=COLUNAS_BASE)
    df = pd.concat(partes, ignore_index=True, sort=False)
    for col in COLUNAS_BASE:
        if col not in df.columns:
            df[col] = pd.NA
    return df[COLUNAS_BASE].copy()


def preservar_dados_manuais_conferidos(base, diagnosticos=None):
    """Preserva A:S da base manual para linhas já conferidas.

    Isso evita regressões de formatação/conteúdo, como perda do zero à esquerda em
    CHAPA, alteração de RPA, ou ajuste indevido de MEDIA_AUDIENCIA.
    """
    base = base.copy()
    diagnosticos = diagnosticos.copy() if isinstance(diagnosticos, pd.DataFrame) else pd.DataFrame()
    referencia = _ler_base_manual_preservacao(PASTA / 'Base_Instituicao_515253.xlsx')
    if referencia.empty:
        resumo = pd.DataFrame([{
            'TIPO_DIAGNOSTICO': 'PRESERVACAO_DADOS_MANUAIS',
            'STATUS': 'NÃO APLICADO - base manual não encontrada',
            'OBSERVAÇÃO': 'Sem Base_Instituicao_515253.xlsx para usar como gabarito.'
        }])
        diagnosticos = pd.concat([diagnosticos, resumo], ignore_index=True, sort=False)
        return base, diagnosticos, resumo

    filas = defaultdict(deque)
    for _, row in referencia.iterrows():
        filas[_chave_preservacao_manual(row)].append(row)

    detalhes = []
    for idx, row in base.iterrows():
        chave = _chave_preservacao_manual(row)
        if not filas.get(chave):
            continue
        ref_row = filas[chave].popleft()
        alteracoes = []
        for col in COLUNAS_PRESERVAR_MANUAL:
            antes = row.get(col, pd.NA)
            depois = ref_row.get(col, pd.NA)
            if pd.isna(depois):
                depois = pd.NA
            if texto_limpo(antes) != texto_limpo(depois):
                alteracoes.append(f'{col}: {antes} -> {depois}')
            base.at[idx, col] = depois
        if alteracoes:
            detalhes.append({
                'TIPO_DIAGNOSTICO': 'PRESERVACAO_DADOS_MANUAIS',
                'STATUS': 'AJUSTADO - colunas A:S preservadas do gabarito',
                'LINHA_BASE_FINAL': int(idx) + 2,
                'MÓDULO': base.at[idx, 'MÓDULO'],
                'CURSO': base.at[idx, 'CURSO'],
                'DISCIPLINA': base.at[idx, 'DISCIPLINA'],
                'ALTERACOES': ' | '.join(alteracoes[:20]),
                'QTD_ALTERACOES': len(alteracoes),
                'OBSERVAÇÃO': 'Colunas A:S foram restauradas conforme a base manual; sinergia T:U permanece enriquecida pela automação.'
            })

    resumo = pd.DataFrame([{
        'TIPO_DIAGNOSTICO': 'PRESERVACAO_DADOS_MANUAIS',
        'STATUS': 'OK - dados manuais preservados antes da exportação',
        'LINHAS_AVALIADAS': len(base),
        'LINHAS_AJUSTADAS': len(detalhes),
        'OBSERVAÇÃO': 'Para linhas existentes na base manual, A:S é gabarito. Linhas novas do 53/54 permanecem automáticas.'
    }])
    diag_preservacao = pd.concat([resumo, pd.DataFrame(detalhes)], ignore_index=True, sort=False)
    diagnosticos = pd.concat([diagnosticos, diag_preservacao], ignore_index=True, sort=False)
    return base, diagnosticos, diag_preservacao


## 4. Exportar Excel e CSV

O Excel exportado contém:

- `GERAL`: base final calibrada com todos os módulos listados em `MODULOS_SAIDA`;
- uma aba por módulo, por exemplo `BASE5126`, `BASE5226`, `BASE5326`;
- `DIAGNOSTICOS`: mostra se a busca foi exata, relaxada, ajustada manualmente ou se houve duplicata integral identificada;
- `DUPLICATAS_IDENTIFICADAS`: lista as duplicatas integrais encontradas sem removê-las ou informa que nada foi encontrado;
- abas auxiliares de calibração, sinergia e ajustes manuais auditáveis.


### Compatibilidade com Microsoft Excel

Após salvar o `.xlsx`, o notebook executa uma limpeza técnica no pacote interno do arquivo para evitar a mensagem do Excel **“Encontramos um problema em um conteúdo”**. Essa etapa não altera dados, fórmulas ou estilos; ela apenas remove marcações internas inválidas de células vazias.


In [ ]:
# Preserva primeiro as colunas A:S das linhas que já existem na base manual.
# Isso mantém CHAPA, R$ (RPA), mediadores e demais dados exatamente como no gabarito.
base, diagnosticos, diagnosticos_preservacao_manual = preservar_dados_manuais_conferidos(base, diagnosticos)
print('Linhas com dados manuais A:S preservados:', len(diagnosticos_preservacao_manual) - 1)

# Preserva valores de mediadores e normaliza formatação dos campos críticos.
# Esta etapa acontece antes da validação e da exportação, para que a base final
# mantenha os mediadores conforme a base manual/revisada.
base, diagnosticos, diagnosticos_mediadores = preservar_mediadores_e_formatacao_dados(base, diagnosticos)
print('Linhas com mediadores preservados/normalizados:', len(diagnosticos_mediadores) - 1)

# Preserva e valida a coluna R$ (RPA) antes da exportação.
base, diagnosticos, diagnosticos_rpa = preservar_rpa_e_formatacao_dados(base, diagnosticos)
print('Linhas com RPA preservado/normalizado:', len(diagnosticos_rpa) - 1)

# Sincroniza R$ (RPA) de módulos complementares com o REGISTRO atualizado.
# Isso corrige o módulo 53 quando a base manual parcial tinha a linha, mas ainda não tinha o valor de RPA.
base, diagnosticos, diagnosticos_rpa_registro = atualizar_rpa_por_registro_modulos(
    base,
    diagnosticos,
    modulos=MODULOS_ATUALIZAR_RPA_PELO_REGISTRO,
)
diagnosticos_rpa = pd.concat([diagnosticos_rpa, diagnosticos_rpa_registro], ignore_index=True, sort=False)
print('Linhas com RPA sincronizado pelo REGISTRO:', max(len(diagnosticos_rpa_registro) - 1, 0))

# Antes de exportar, trava os totais dos módulos já validados.
# Isso evita que uma execução sem base conferida gere o 51 automaticamente com total incorreto.
validar_totais_calibrados(
    base,
    totais_validados=TOTAIS_ALUNOS_VALIDADOS,
    linhas_validadas=LINHAS_VALIDADAS,
    modulos_saida=MODULOS_SAIDA,
    etapa='pré-exportação',
)

# Verifica duplicatas integrais, mas NÃO remove nenhuma linha.
# A base final deve permanecer conforme o gabarito/manual.
base, diagnosticos, diagnosticos_duplicatas = verificar_duplicatas_inteiras_sem_remover(base, diagnosticos)
print('Duplicatas integrais identificadas e mantidas:', len(diagnosticos_duplicatas))

# Exportação otimizada: escreve as abas em uma única rotina openpyxl direta.
# Isso evita reabrir o arquivo várias vezes e mantém a formatação da base manual.
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from copy import copy as _copy_style

COLUNAS_EXPORTACAO = [c for c in COLUNAS_BASE if c in base.columns]
base_export = base[COLUNAS_EXPORTACAO].copy()
base_export.to_csv(SAIDA_CSV, index=False, encoding='utf-8-sig')

# Tenta usar a planilha manual como template visual. Se não existir, usa estilo verde padrão.
template_ws = None
for candidato_template in [PASTA / 'Base_Instituicao_515253.xlsx', ARQUIVO_BASE_HISTORICA]:
    try:
        if candidato_template and Path(candidato_template).exists():
            wb_template = load_workbook(candidato_template)
            if 'BASE5126' in wb_template.sheetnames:
                template_ws = wb_template['BASE5126']
                break
    except Exception:
        template_ws = None

wb_saida = Workbook()
wb_saida.remove(wb_saida.active)

header_fill_padrao = PatternFill('solid', fgColor='70AD47')
header_font_padrao = Font(color='FFFFFF', bold=True)
thin_green = Side(style='thin', color='A9D18E')
borda_padrao = Border(left=thin_green, right=thin_green, top=thin_green, bottom=thin_green)

if template_ws is not None:
    largura_template = {get_column_letter(i): template_ws.column_dimensions[get_column_letter(i)].width for i in range(1, len(COLUNAS_EXPORTACAO)+1)}
    header_style = [_copy_style(template_ws.cell(1, i)._style) for i in range(1, len(COLUNAS_EXPORTACAO)+1)]
    odd_style = [_copy_style(template_ws.cell(2, i)._style) for i in range(1, len(COLUNAS_EXPORTACAO)+1)]
    even_style = [_copy_style(template_ws.cell(3, i)._style) for i in range(1, len(COLUNAS_EXPORTACAO)+1)]
else:
    largura_template = {}
    header_style = None
    odd_style = None
    even_style = None


def _valor_excel(v):
    if isinstance(v, (dict, list, tuple, set)):
        return str(v)
    try:
        if pd.isna(v):
            return ''
    except Exception:
        pass
    if isinstance(v, np.generic):
        return v.item()
    return v


def _escrever_df(ws, df, aplicar_template=False):
    df = df.copy()
    # Cabeçalho
    for j, col in enumerate(df.columns, start=1):
        cell = ws.cell(1, j, col)
        if aplicar_template and header_style is not None and j <= len(header_style):
            cell._style = _copy_style(header_style[j-1])
        else:
            cell.fill = header_fill_padrao
            cell.font = header_font_padrao
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = borda_padrao
        # Força o padrão verde do gabarito manual, mesmo quando o Excel aplica tema de tabela.
        cell.fill = header_fill_padrao
        cell.font = Font(color='FFFFFF', bold=True, name='Calibri', size=8)
    # Dados
    for i, row in enumerate(df.itertuples(index=False, name=None), start=2):
        style_src = None
        if aplicar_template and odd_style is not None and even_style is not None:
            style_src = odd_style if i % 2 == 0 else even_style
        for j, value in enumerate(row, start=1):
            col = df.columns[j-1]
            cell = ws.cell(i, j, _valor_excel(value))
            if style_src is not None and j <= len(style_src):
                cell._style = _copy_style(style_src[j-1])
            # Força listras verde claro/branco no mesmo padrão visual da planilha manual.
            cell.fill = PatternFill('solid', fgColor='E2F0D9' if i % 2 == 0 else 'FFFFFF')
            cell.font = Font(color='000000', name='Calibri', size=8)
            cell.border = borda_padrao
            if col in {'CHAPA', 'CHAPA2', 'MÓDULO'}:
                cell.number_format = '@'
                cell.value = '' if cell.value == '' else str(cell.value)
            elif col in {'QTD_ALUNOS','CH SEMANAL','CH LETIVA','CH_SEMANAL','MEDIA_AUDIENCIA'}:
                cell.number_format = '0'
            elif col == 'R$ (RPA)':
                cell.number_format = '0.##'
    ws.freeze_panes = 'A2'
    if ws.max_row >= 1 and ws.max_column >= 1:
        ws.auto_filter.ref = ws.dimensions
    for j in range(1, ws.max_column + 1):
        col_letter = get_column_letter(j)
        if aplicar_template and largura_template.get(col_letter):
            ws.column_dimensions[col_letter].width = largura_template[col_letter]
        else:
            max_len = 0
            for r in range(1, min(ws.max_row, 200) + 1):
                max_len = max(max_len, len(str(ws.cell(r, j).value or '')))
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 48)
    ws.row_dimensions[1].height = 18
    if aplicar_template and ws.max_row >= 2 and ws.max_column >= 1:
        # Aplica o mesmo comportamento visual de tabela da base manual.
        # A base manual usa o estilo verde TableStyleMedium7.
        try:
            nome_tabela = 'Tabela_' + re.sub(r'[^A-Za-z0-9_]', '_', ws.title)
            ref_tabela = f'A1:{get_column_letter(ws.max_column)}{ws.max_row}'
            tabela = Table(displayName=nome_tabela[:254], ref=ref_tabela)
            estilo = TableStyleInfo(
                name='TableStyleMedium7',
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False,
            )
            tabela.tableStyleInfo = estilo
            ws.add_table(tabela)
        except Exception:
            pass


def _add_sheet(nome, df, aplicar_template=False):
    nome = str(nome)[:31]
    nome_final = nome
    n = 1
    while nome_final in wb_saida.sheetnames:
        sufixo = f'_{n}'
        nome_final = nome[:31-len(sufixo)] + sufixo
        n += 1
    ws = wb_saida.create_sheet(nome_final)
    if df is None or len(df) == 0:
        df = pd.DataFrame([{'STATUS': 'Sem registros nesta execução.'}])
    _escrever_df(ws, df, aplicar_template=aplicar_template)
    return ws

# Abas principais.
_add_sheet('GERAL', base_export, aplicar_template=True)
for modulo, df_mod in base_export.groupby('MÓDULO', dropna=False):
    _add_sheet(nome_aba_modulo(modulo), df_mod, aplicar_template=True)

# Abas de auditoria e apoio.
resumo = resumo_validacao(base_export, diagnosticos, parametros_processo, base_historica=ARQUIVO_BASE_HISTORICA)
_add_sheet('DIAGNOSTICOS', diagnosticos)
_add_sheet('PARAMETROS_PROCESSO', parametros_processo)
_add_sheet('RESUMO_VALIDACAO', resumo)
_add_sheet('INCREMENTOS_AUTOMATICOS', incrementos_calibracao[[c for c in COLUNAS_BASE if c in incrementos_calibracao.columns]] if isinstance(incrementos_calibracao, pd.DataFrame) else pd.DataFrame())
_add_sheet('COMPARATIVO_CALIBRACAO', comparativo_calibracao)
_add_sheet('DIVERGENCIAS_AUTO_MANUAL', divergencias_auto_manual)
_add_sheet('DIAG_SINERGIA', diagnosticos_sinergia if isinstance(diagnosticos_sinergia, pd.DataFrame) and not diagnosticos_sinergia.empty else pd.DataFrame([{'STATUS': 'Todas as linhas ficaram com CURSOS_SINERGIA e CLUSTERS_SINERGIA preenchidos.'}]))
_add_sheet('AJUSTES_ALUNOS', diagnosticos_ajustes_alunos if isinstance(diagnosticos_ajustes_alunos, pd.DataFrame) and not diagnosticos_ajustes_alunos.empty else pd.DataFrame([{'STATUS': 'Nenhum ajuste manual de alunos aplicado.'}]))
_add_sheet('AJUSTES_CLUSTER', diagnosticos_cluster_principal if isinstance(diagnosticos_cluster_principal, pd.DataFrame) and not diagnosticos_cluster_principal.empty else pd.DataFrame([{'STATUS': 'Nenhum CLUSTER principal precisou ser preenchido.'}]))
_add_sheet('DUPLICATAS_IDENTIFICADAS', diagnosticos_duplicatas if isinstance(diagnosticos_duplicatas, pd.DataFrame) and not diagnosticos_duplicatas.empty else pd.DataFrame([{'STATUS': 'Nenhuma duplicata integral encontrada. Nenhuma linha removida.'}]))
_add_sheet('CONTROLE_DADOS_MANUAIS', diagnosticos_preservacao_manual)
_add_sheet('CONTROLE_MEDIADORES', diagnosticos_mediadores)
_add_sheet('CONTROLE_RPA', diagnosticos_rpa)


# -----------------------------------------------------------------------------
# Correção preventiva de compatibilidade com Microsoft Excel
# -----------------------------------------------------------------------------
# Algumas versões do Excel exibem a mensagem "Encontramos um problema em um conteúdo"
# quando o arquivo XLSX possui células vazias gravadas internamente como
# inlineStr sem o nó <is>. Visualmente o arquivo costuma abrir após "recuperar",
# mas isso não é aceitável para uso operacional.
#
# A função abaixo limpa somente essa representação interna inválida de células
# vazias, mantendo valores, estilos, abas, tabelas e filtros.
def sanitizar_xlsx_para_excel(caminho_xlsx):
    import os
    import zipfile
    import xml.etree.ElementTree as ET

    namespace_planilha = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'
    namespace_rel = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
    ET.register_namespace('', namespace_planilha)
    ET.register_namespace('r', namespace_rel)

    caminho_tmp = str(caminho_xlsx) + '.tmp'
    total_corrigido = 0

    with zipfile.ZipFile(caminho_xlsx, 'r') as entrada, zipfile.ZipFile(caminho_tmp, 'w', zipfile.ZIP_DEFLATED) as saida:
        for item in entrada.infolist():
            conteudo = entrada.read(item.filename)

            if item.filename.startswith('xl/worksheets/sheet') and item.filename.endswith('.xml'):
                raiz = ET.fromstring(conteudo)
                corrigido_na_aba = 0

                for celula in raiz.iter(f'{{{namespace_planilha}}}c'):
                    if celula.attrib.get('t') == 'inlineStr':
                        tem_inline_string = celula.find(f'{{{namespace_planilha}}}is') is not None
                        tem_valor = celula.find(f'{{{namespace_planilha}}}v') is not None

                        # Célula vazia com tipo inlineStr, mas sem conteúdo.
                        # O Excel pode interpretar isso como conteúdo inválido.
                        if not tem_inline_string and not tem_valor and len(list(celula)) == 0:
                            celula.attrib.pop('t', None)
                            corrigido_na_aba += 1

                if corrigido_na_aba:
                    conteudo = ET.tostring(raiz, encoding='utf-8', xml_declaration=False)
                    total_corrigido += corrigido_na_aba

            info = zipfile.ZipInfo(item.filename, date_time=item.date_time)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = item.external_attr
            saida.writestr(info, conteudo)

    os.replace(caminho_tmp, caminho_xlsx)
    return total_corrigido


wb_saida.save(SAIDA_XLSX)
celulas_xlsx_sanitizadas = sanitizar_xlsx_para_excel(SAIDA_XLSX)
print(f'XLSX sanitizado para Excel: {celulas_xlsx_sanitizadas} células vazias internas corrigidas.')

print('Arquivos exportados:')
print('-', SAIDA_XLSX)
print('-', SAIDA_CSV)
resumo


## Checklist para próximos módulos

Antes de rodar um novo módulo, confira:

- O arquivo `FLUXO` contém o ano/semestre do módulo novo.
- O `MAPA PEDAGÓGICO` do módulo novo está na pasta.
- O `REGISTRO_PROFESSORES` contém as linhas do módulo novo.
- `MODULOS_SAIDA` contém todos os módulos que você quer no Excel final.
- `MODULOS_PROCESSAR_AUTO` contém apenas os mapas que você quer processar agora.
- Se `MODULOS_SAIDA` tiver módulos que não estão em `MODULOS_PROCESSAR_AUTO`, mantenha a base conferida na pasta.
- Depois de rodar, confira a contagem por módulo impressa na última célula e a aba `DIAGNOSTICOS`.

Para manter 51/52/53 exatamente como a base conferida/manual, use `MODULOS_PROCESSAR_AUTO = []`.

Para acrescentar o 54 sem perder 51/52/53, a configuração recomendada é:

```python
MODULOS_SAIDA = [51, 52, 53, 54]
MODULOS_PROCESSAR_AUTO = [54]
MODULOS_FECHADOS_BASE_CONFERIDA = [51, 52, 53]
MODULOS_COMPLEMENTAR_BASE_CONFERIDA = []
```


## 5. Conferências rápidas antes de usar no dashboard

Esta última seção imprime verificações básicas:

- colunas finais da base;
- quantidade de linhas por módulo;
- linhas com `QTD_ALUNOS` zerada;
- preenchimento de `CURSOS_SINERGIA` e `CLUSTERS_SINERGIA`;
- ajustes manuais aplicados.

Se algum indicador aparecer inconsistente, comece a investigação pela aba `DIAGNOSTICOS` do Excel exportado.


In [ ]:
# Conferência final. Use esta saída antes de entregar a base ao dashboard.
# Critérios esperados nesta versão:
# - nenhuma linha com QTD_ALUNOS zerada;
# - CURSOS_SINERGIA e CLUSTERS_SINERGIA preenchidos em todas as linhas;
# - módulos com contagens coerentes com o comparativo de calibração;
# - duplicatas integrais apenas identificadas, sem remoção.
print('Colunas da base:')
print(list(base.columns))

print('\nLinhas por módulo:')
print(base['MÓDULO'].value_counts(dropna=False).sort_index().to_string())

print('\nQuantidade de linhas com QTD_ALUNOS zerada por módulo:')
print((base.assign(QTD_ALUNOS_NUM=pd.to_numeric(base['QTD_ALUNOS'], errors='coerce').fillna(0))
          .groupby('MÓDULO')['QTD_ALUNOS_NUM']
          .apply(lambda s: int((s == 0).sum()))
          .to_string()))

print('\nResumo da calibração:')
display(comparativo_calibracao)

print('\nPrimeiros incrementos automáticos adicionados:')
display(incrementos_calibracao[COLUNAS_BASE].head(20))

print('\nPreenchimento de sinergia:')
print('CURSOS_SINERGIA preenchidos:', int(base['CURSOS_SINERGIA'].map(lambda x: bool(texto_limpo(x))).sum()), '/', len(base))
print('CLUSTERS_SINERGIA preenchidos:', int(base['CLUSTERS_SINERGIA'].map(lambda x: bool(texto_limpo(x))).sum()), '/', len(base))

print('\nAjustes manuais de alunos aplicados:')
print(diagnosticos_ajustes_alunos.to_string(index=False) if not diagnosticos_ajustes_alunos.empty else 'Nenhum ajuste aplicado.')

print('\nValidação contra totais conferidos:')
base['_MOD_VALIDACAO'] = base['MÓDULO'].map(extrair_modulo_num)
base['_QTD_ALUNOS_NUM'] = pd.to_numeric(base['QTD_ALUNOS'], errors='coerce').fillna(0)
for mod, total_esperado in TOTAIS_ALUNOS_VALIDADOS.items():
    if mod in MODULOS_SAIDA:
        df_mod = base[base['_MOD_VALIDACAO'].eq(mod)]
        total_atual = int(df_mod['_QTD_ALUNOS_NUM'].sum())
        linhas_atual = int(len(df_mod))
        linhas_esperadas = LINHAS_VALIDADAS.get(mod)
        status_total = 'OK' if total_atual == int(total_esperado) else f'REVISAR: esperado {int(total_esperado)}'
        status_linhas = 'OK' if linhas_esperadas is None or linhas_atual == int(linhas_esperadas) else f'REVISAR: esperado {int(linhas_esperadas)}'
        print(f'Módulo {mod}: alunos={total_atual} ({status_total}) | linhas={linhas_atual} ({status_linhas})')

cluster_vazio = int(base['CLUSTER'].map(lambda x: not bool(texto_limpo(x))).sum())
if cluster_vazio:
    raise ValueError(f'Existem {cluster_vazio} linhas com CLUSTER vazio. Verifique AJUSTES_CLUSTER e a base conferida.')

base = base.drop(columns=['_MOD_VALIDACAO','_QTD_ALUNOS_NUM'], errors='ignore')

print('\nDuplicatas integrais identificadas e mantidas na base final:')
print(int(base.duplicated(subset=list(base.columns), keep='first').sum()))

print('\nDuplicatas removidas nesta execução:')
print(0)


print('
Resumo RPA por módulo para Power BI:')
base['_RPA_NUM'] = base[RPA_COLUNA].map(_numero_rpa_seguro)
print((base.groupby('MÓDULO')['_RPA_NUM']
          .agg(qtd_rpa_com_valor=lambda s: int(s.map(lambda x: not pd.isna(x)).sum()),
               total_rpa=lambda s: round(float(pd.to_numeric(s, errors='coerce').fillna(0).sum()), 2))
          .to_string()))
base = base.drop(columns=['_RPA_NUM'], errors='ignore')


## Manutenção e evolução

Principais pontos de manutenção futura:

- **Nova estrutura no fluxo:** revisar `carregar_fluxo`.
- **Nova estrutura no registro:** revisar `carregar_registro`.
- **Nova aba/padrão no mapa pedagógico:** revisar `detectar_aba_mapa`, `carregar_mapa` e `selecionar_candidatos_mapa`.
- **Mudanças em monitoramento/mediadores:** revisar `detectar_aba_monitoramento`, `carregar_monitoramento` e `enriquecer_mediador`.
- **Exceções de negócio:** preferir registrar em `AJUSTES_MANUAIS_SINERGIA` ou `AJUSTES_MANUAIS_ALUNOS`, pois isso mantém rastreabilidade na aba `DIAGNOSTICOS`.
- **Duplicatas:** a verificação é automática apenas para linhas inteiras iguais. As duplicatas são mantidas conforme a base conferida/manual e aparecem em `DIAGNOSTICOS` para auditoria.

A regra geral é: **automação para escala, base manual para calibração, diagnósticos para confiança**.
